In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Unsupervised Tissue Macro-Structure Classification & Spatial Annotation
## CODEX Multiplex Imaging Pipeline — High-Resolution Spatial Microenvironment Discovery

**Overview:** 
This notebook performs automated unsupervised macro-structure discovery and tissue microenvironment annotation on highly multiplexed CODEX imaging datasets.

Key computational steps include:
1. **Pyramidal OME-TIFF Processing**: Rapid anti-aliased multi-resolution data loading.
2. **Dual-Mode Tissue Boundary Masking**: Automated H&E luminance thresholding combined with multiplex CODEX signal projection.
3. **Automated Blur Artifact Subtraction**: OpenCV-accelerated normalized Laplacian focus scoring to subtract out-of-focus signal.
4. **Bidirectional QuPath GeoJSON Integration**: Seamless export and re-import of tissue annotations for interactive user verification.
5. **Dynamic Arcsinh Normalization**: Marker-specific background noise estimation and Z-score feature matrix construction.
6. **Unsupervised Microenvironment Discovery**: MiniBatchKMeans spatial phenotype clustering with high-contrast qualitative color assignment.
7. **Organic Macro-Scale Vector Export**: Polygon boundary extraction with quadtree tile aggregation, Gaussian smoothing, and MultiPolygon GeoJSON export.

In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
import os
import sys
import json
import uuid
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

import tifffile
import cv2
from scipy.ndimage import gaussian_filter, uniform_filter, binary_fill_holes
from skimage.filters import gaussian, threshold_otsu
from skimage.morphology import (
    binary_closing, binary_opening, binary_dilation,
    disk, remove_small_objects, remove_small_holes
)
from skimage.measure import find_contours

from shapely.geometry import Polygon, MultiPolygon, shape, mapping
from shapely.ops import unary_union
from shapely.validation import make_valid

import rasterio.features

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
})
sns.set_style('whitegrid')

print("All libraries imported successfully.")

## Configuration Overview

**Dual-Resolution Architecture Settings:**
- **`BLUR_DOWNSAMPLE = 2`**: 2x spatial downsampling for high-precision tissue boundary, blur, and lint masking.
- **`CLUSTERING_DOWNSAMPLE = 4`**: 4x clustering grid and high-resolution GeoJSON export grid, approximately 2.03 µm per raster pixel.
- **`MACRO_EXPORT_MODE = "fine_gapless"`**: Sequential winner-take-all smoothing at the clustering grid; every valid-tissue pixel receives exactly one class.
- **`FINE_EXPORT_SMOOTH_SIGMA = 2.0`**: Controls spatial regularization without reducing export resolution or holding all class score maps in memory.
- **`VALID_TISSUE_EXCLUDED_CLUSTERS`**: Cluster IDs marked as residual lint; all remaining labels are merged into a separate high-resolution valid-tissue GeoJSON.

In [ ]:
# ============================================================================
# CELL 3: GLOBAL CONFIGURATION & FILE PATHS (Dual Resolution & Macro Scaling)
# ============================================================================

import os
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# ----- File Paths -----
ROI_DIR    = f"{DATA_DIR}/example_study/registration/roi/" #oph_slide5_merged.ome_bg_subtracted.ome.tiff
OUTPUT_DIR = f"{DATA_DIR}/example_study/registration/roi/tissue_annotation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_ID  = "P28_roi_1"  # e.g., "s1r1", "s4r2", "s10r1"

# ----- Spatial & Reproducibility Settings -----
BLUR_DOWNSAMPLE       = 2   # 2x downsample for high-precision masking
CLUSTERING_DOWNSAMPLE = 4   # 4x downsample for 35-channel feature clustering
RANDOM_SEED           = 42  # Reproducibility seed
MICRONS_PER_PIXEL     = 0.5083  # Pixel size in microns (CODEX default)

# ----- Channels to EXCLUDE from loading & clustering -----
EXCLUDE_CHANNELS = [
    # "DAPI",  # Uncomment to exclude DAPI from clustering
    "HE_Red", "HE_Green", "HE_Blue",  # Virtual H&E channels
]

# ----- Signal Transformation & Feature Engineering Settings -----
ARCSINH_MODE            = "dynamic"  # "fixed" (constant cofactor) or "dynamic" (per-marker tissue noise estimation)
ARCSINH_COFACTOR        = 5.0        # Fixed arcsinh cofactor (used when ARCSINH_MODE = "fixed")
# ARCSINH_DYNAMIC_K sets the signal-to-noise floor threshold per channel.
# Example: If a marker's background noise std-dev is 2.0, and k=3.0, the cofactor becomes max(5.0, 3.0 * 2.0) = 6.0.
# Any pixel intensity below 6.0 is aggressively compressed (treated as noise), while values above 6.0 scale more linearly.
ARCSINH_DYNAMIC_K       = 3.0        # Multiplier k for background noise std (c_j = max(min_val, k * sigma_bg))
ARCSINH_DYNAMIC_MIN     = 5.0        # Minimum floor value for dynamic cofactor
PERCENTILE_THRESHOLD    = 99.9       # Per-channel upper percentile used to clip hot pixels
SIGMA_SPATIAL_SMOOTHING = 0.8        # Spatial feature Gaussian smoothing (sigma); 0.0 disables it

# ----- Feature Selection & Low Signal Handling -----
USE_TOP_K_MASKING    = True       # True for Top-K masking; False for continuous Z-score features
# TOP_K_MODE controls how many top-enriched markers are kept per pixel (see Step 1.5):
#   "auto"   : data-driven elbow detection on the mean sorted Z-score profile of the actual tissue (recommended
#              starting point — adapts K to how many markers are typically co-enriched in THIS sample/panel).
#   "manual" : always use the fixed TOP_K_MARKERS value below, ignoring the data-driven estimate.
TOP_K_MODE           = "auto"     # "auto" or "manual"
TOP_K_MARKERS        = 5          # Manual K. Used directly when TOP_K_MODE="manual"; used as a fallback if "auto" detection fails.
TOP_K_AUTO_MIN        = 2         # Auto mode: smallest K the elbow search is allowed to pick
TOP_K_AUTO_MAX        = 10        # Auto mode: largest K the elbow search is allowed to pick
LOW_SIGNAL_HANDLING  = "include"  # "include" or "reserve"
LOW_SIGNAL_QUANTILE  = 0.05       # Lowest fraction of tissue set aside when handling="reserve"

# ----- Unsupervised Clustering Settings -----
N_CLUSTERS           = 20
KMEANS_BATCH_SIZE    = 2048
KMEANS_N_INIT        = 5
KMEANS_MAX_ITER      = 300

# ----- Macro-Scale Organic GeoJSON Export Parameters -----
MACRO_TILE_SIZE           = 2     # 4 = 4x4 tile aggregation for organic macro scaling
MACRO_SMOOTH_SIGMA        = 1.5   # Gaussian smoothing sigma: turns 90-deg pixel steps into smooth organic curves
MACRO_MIN_TISSUE_FRACTION = 0.25  # Tile must be at least 25% tissue
MAX_RAW_POLYGONS          = 20000 # Safety threshold
BOUNDARY_SIMPLIFY_UM      = 1.5   # Douglas-Peucker tolerance (um): drops redundant/near-colinear vertices; main file-size control
# BOUNDARY_SMOOTH_UM rounds the coarse tile-grid staircase into organic curves via a buffer(+r).buffer(-r) round-trip,
# applied ONCE to the final per-class merged geometry (not per raw tile polygon). This decouples "prettiness" from
# MACRO_TILE_SIZE: you get smooth organic boundaries without shrinking the tile size (which is what previously made
# file sizes balloon). Set to 0 to disable and keep the raw tile-stepped boundary.
BOUNDARY_SMOOTH_UM        = 3.0   # Corner-rounding radius (um). Larger = more organic/rounded, but can erode thin features.
BOUNDARY_SMOOTH_QUAD_SEGS = 4     # Arc resolution for the round-trip buffer (points per quarter-circle). Lower = smaller files.
MERGE_GEOMETRIES          = True  # True: exports 1 unified MultiPolygon feature per cluster

HE_CHANNEL_PATTERNS = ["he_", "he ", "h&e", "he_red", "he_green", "he_blue"]

# ----- Display & Seaborn Settings -----
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
})
sns.set_style("whitegrid")

# ----- Color & Annotation Settings -----
AUTO_COLOR = True               # True: auto-assign distinctive deterministic colors per class; False: use CLASS_COLORS
CLASS_COLORS = {
    "Epithelia": -65536,          # Red
    "Stroma": -16711936,         # Green
    "Immune": -256,              # Yellow
    "Myeloid": -16776961,        # Blue
    "Low_Signal_Tissue": -7829368,  # Gray
    "Background": -7829368       # Gray
}

def get_class_color(name, seed=RANDOM_SEED, auto_color=AUTO_COLOR, manual_colors=CLASS_COLORS):
    # Returns signed 32-bit ARGB integer for QuPath for a given class name.
    if manual_colors is None: manual_colors = CLASS_COLORS
    if not auto_color and name in manual_colors: return manual_colors[name]
    if name in ["Low_Signal_Tissue", "Background", "Non-Tissue"]:
        R, G, B = 136, 136, 136
    elif auto_color is False and name in manual_colors:
        return manual_colors[name]
    else:
        hash_bytes = hashlib.md5(f"{seed}_{name}".encode('utf-8')).digest()
        hash_int = int.from_bytes(hash_bytes[:4], 'big')
        hue = (hash_int % 360) / 360.0
        sat = 0.75 + 0.25 * ((hash_int >> 8) % 100) / 100.0
        val = 0.85 + 0.15 * ((hash_int >> 16) % 100) / 100.0
        rgb = mcolors.hsv_to_rgb((hue, sat, val))
        R, G, B = int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255)
    unsigned_val = (0xFF << 24) | (R << 16) | (G << 8) | B
    return unsigned_val if unsigned_val < 0x80000000 else unsigned_val - 0x100000000

def get_class_hex_color(name, seed=RANDOM_SEED, auto_color=AUTO_COLOR, manual_colors=CLASS_COLORS):
    # Returns hex color string '#RRGGBB' for matplotlib plotting.
    argb_val = get_class_color(name, seed=seed, auto_color=auto_color, manual_colors=manual_colors)
    rgb_24bit = argb_val & 0xFFFFFF
    return f"#{rgb_24bit:06x}"

tiff_path = os.path.join(ROI_DIR, f"{SAMPLE_ID}.ome.tiff")
print("Global configuration initialized (Organic Macro Scaling Mode).")
print(f"  Sample ID             : {SAMPLE_ID}")
print(f"  Blur Downsample       : {BLUR_DOWNSAMPLE}x (High spatial resolution)")
print(f"  Clustering Downsample : {CLUSTERING_DOWNSAMPLE}x (Optimized feature space)")
print(f"  Arcsinh Mode          : {ARCSINH_MODE} (Fixed cofactor={ARCSINH_COFACTOR}, Dynamic k={ARCSINH_DYNAMIC_K})")
if USE_TOP_K_MASKING:
    print(f"  Top-K Marker Mode     : {TOP_K_MODE} (manual fallback K={TOP_K_MARKERS}, auto search range=[{TOP_K_AUTO_MIN},{TOP_K_AUTO_MAX}])")
else:
    print("  Top-K Marker Mode     : disabled (continuous Z-score features)")
print(f"  Macro Tile Size       : {MACRO_TILE_SIZE}x4x (Organic MultiPolygon GeoJSON export)")
print(f"  Boundary Polish       : simplify={BOUNDARY_SIMPLIFY_UM}um, smooth_radius={BOUNDARY_SMOOTH_UM}um (applied post-merge, size-safe)")


---
# Phase 1: Pre-processing & Feature Engineering

### Step 1.1 — Multichannel OME-TIFF Pyramid Loading & Metadata Extraction

**Objective:** Read and inspect multi-resolution OME-TIFF image headers. Lower resolution pyramid levels are loaded directly for anti-aliased downsampling without requiring full-resolution memory allocation.

In [ ]:
# ============================================================================
# STEP 1.1: Load Multichannel OME-TIFF & Metadata (Pyramid-Aware Reader)
# ============================================================================
# PERFORMANCE: These OME-TIFF files are pyramidal (multi-resolution).
# Each channel has 4 resolution levels: Level 0 (full), Level 1 (2x), Level 2 (4x), Level 3 (8x).
# Series mapping: series_index = channel_index * n_levels + level_index
# Reading the pyramid level directly is 5x+ faster than reading full-res + striding,
# AND produces anti-aliased (block-averaged) data which is higher quality than strided sampling.

assert os.path.exists(tiff_path), f"File not found: {tiff_path}"

with tifffile.TiffFile(tiff_path) as tif:
    ome_xml = tif.ome_metadata
    root = ET.fromstring(ome_xml)
    ns = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
    all_channel_names = [
        ch.get('Name', f'Channel_{i}')
        for i, ch in enumerate(root.findall('.//ome:Channel', ns))
    ]
    n_channels = len(all_channel_names)
    n_series = len(tif.series)
    n_levels_per_channel = n_series // n_channels if n_channels > 0 else 1

    # Detect pyramid: if n_series > n_channels, the file is pyramidal
    IS_PYRAMIDAL = n_levels_per_channel > 1
    if IS_PYRAMIDAL:
        print(f"  ✓ Pyramidal OME-TIFF detected: {n_channels} channels × {n_levels_per_channel} resolution levels")
        for li in range(n_levels_per_channel):
            s = tif.series[li]  # channel 0, level li
            print(f"    Level {li}: {s.shape[0]}×{s.shape[1]} ({s.shape[0]*s.shape[1]:,} px)")
    else:
        print(f"  Non-pyramidal OME-TIFF: {n_channels} channels, will use strided reads")

# --- Pyramid-aware reader: reads the correct resolution level directly ---
# factor=1 → Level 0 (full res), factor=2 → Level 1 (2x), factor=4 → Level 2 (4x), etc.
# Falls back to strided read for non-pyramidal files.
def read_pyramid_level(tiff_file, channel_idx, factor):
    '''Read a specific channel at the requested downsample factor.
    Uses pyramid level directly if available (5x+ faster, anti-aliased).
    Falls back to strided read for non-pyramidal files.'''
    if IS_PYRAMIDAL and factor > 1:
        level_idx = int(np.log2(factor))  # factor=2→level1, factor=4→level2
        level_idx = min(level_idx, n_levels_per_channel - 1)
        series_idx = channel_idx * n_levels_per_channel + level_idx
        return tiff_file.series[series_idx].asarray().astype(np.float32)
    else:
        # Fallback: read full-res page and stride
        return tiff_file.pages[channel_idx].asarray()[::factor, ::factor].astype(np.float32)

print(f"Loaded OME-TIFF: {SAMPLE_ID}.ome.tiff ({n_channels} channels, {n_levels_per_channel} levels).")

### Step 1.2 — H&E Optical-Density Tissue Masking & Initial Boundary Export

**Objective:** Separate histologic tissue from slide background using the virtual H&E channels only. Each H&E channel is normalized to a channel-specific white-background reference, converted to optical density, and combined into a stain-strength map before Otsu thresholding. Zero-valued acquisition borders are excluded explicitly, preventing black scanner frames and CODEX-only lint from entering the tissue mask. An optional CODEX edge-rescue mode remains available for datasets with incomplete H&E coverage.

In [ ]:
# ============================================================================
# STEP 1.2: Robust H&E Optical-Density Tissue Masking (2x Downsample Precision)
# ============================================================================

import time
t_s12_start = time.time()

CLAMP_SIGMA = 4.0
TISSUE_THRESHOLD     = 10.0
TISSUE_MORPH_RADIUS  = 11
MIN_TISSUE_ISLAND_PX = 5000

# Normalize each virtual H&E channel to its own white-background reference.
HE_WHITE_PERCENTILE    = 99.5
HE_OD_THRESHOLD_FACTOR = 0.85
HE_VALID_MIN_INTENSITY = 0.0

# "he_only" prevents CODEX-only lint from becoming tissue.
# "he_with_codex_edge_rescue" permits CODEX only near an H&E-positive boundary.
# "he_codex_union" reproduces the previous permissive behavior.
TISSUE_MASK_MODE             = "he_only"
HE_CODEX_EDGE_RESCUE_MICRONS = 20.0
TISSUE_BORDER_EXCLUSION_UM   = 20.0

BRIGHT_BREADTH_CHANNEL_SIGMA = 3.0
ROBUST_SIGNAL_CLIP_Z         = 50.0

def clamp_and_normalize(arr_f32, sigma=CLAMP_SIGMA):
    mean, std = arr_f32.mean(), arr_f32.std()
    arr_f32 = np.clip(arr_f32, 0, mean + sigma * std)
    low, high = arr_f32.min(), arr_f32.max()
    if high > low:
        arr_f32 = (arr_f32 - low) / (high - low) * 255.0
    return arr_f32.astype(np.float32)

def _is_he(name):
    name_lower = name.lower()
    return any(name_lower.startswith(pattern) or name_lower == pattern.strip('_') for pattern in HE_CHANNEL_PATTERNS)

def _robust_channel_z(arr_f32):
    sample = arr_f32[::4, ::4]
    sample = sample[np.isfinite(sample) & (sample > 0)]
    if sample.size == 0:
        return np.zeros_like(arr_f32, dtype=np.float32)
    median = float(np.median(sample))
    mad = float(np.median(np.abs(sample - median)))
    robust_sigma = 1.4826 * mad
    if robust_sigma < 1e-6:
        robust_sigma = max(float(sample.std()), 1.0)
    return np.clip((arr_f32 - median) / robust_sigma, 0.0, ROBUST_SIGNAL_CLIP_Z).astype(np.float32)

he_indices = [index for index, name in enumerate(all_channel_names) if _is_he(name)]
codex_indices = [index for index, name in enumerate(all_channel_names) if not _is_he(name)]

he_channels_2x = []
codex_proj_2x = None
codex_robust_max_2x = None
channel_positive_count_2x = None

with tifffile.TiffFile(tiff_path) as tif:
    for index in range(n_channels):
        channel = read_pyramid_level(tif, index, BLUR_DOWNSAMPLE)
        if index in codex_indices:
            robust_z = _robust_channel_z(channel)
            strongly_positive = robust_z >= BRIGHT_BREADTH_CHANNEL_SIGMA
            if channel_positive_count_2x is None:
                channel_positive_count_2x = strongly_positive.astype(np.uint8)
                codex_robust_max_2x = robust_z
            else:
                channel_positive_count_2x += strongly_positive.astype(np.uint8)
                np.maximum(codex_robust_max_2x, robust_z, out=codex_robust_max_2x)

            if codex_proj_2x is None:
                codex_proj_2x = channel
            else:
                np.maximum(codex_proj_2x, channel, out=codex_proj_2x)
        else:
            he_channels_2x.append(channel)

t_io = time.time()
print(f"  [Timing] TIFF Channel I/O (Level 1, {n_channels} channels): {t_io - t_s12_start:.1f}s")

codex_proj_2x = clamp_and_normalize(codex_proj_2x, CLAMP_SIGMA)

he_gray_2x = None
he_stain_score_2x = None
he_valid_pixels_2x = None
he_white_references = []
he_rgb_display = None

if len(he_channels_2x) >= 3:
    he_stain_score_2x = np.zeros_like(he_channels_2x[0], dtype=np.float32)
    he_valid_pixels_2x = np.ones_like(he_channels_2x[0], dtype=bool)

    for channel in he_channels_2x[:3]:
        sample = channel[::4, ::4]
        positive_sample = sample[np.isfinite(sample) & (sample > HE_VALID_MIN_INTENSITY)]
        white_reference = float(np.percentile(positive_sample, HE_WHITE_PERCENTILE)) if positive_sample.size else 1.0
        white_reference = max(white_reference, 1.0)
        he_white_references.append(white_reference)
        he_valid_pixels_2x &= np.isfinite(channel) & (channel > HE_VALID_MIN_INTENSITY)
        normalized_channel = np.clip((channel + 1.0) / (white_reference + 1.0), 1e-4, 1.0)
        he_stain_score_2x += -np.log(normalized_channel)

    he_stain_score_2x /= 3.0
    stain_scale = np.percentile(he_stain_score_2x[he_valid_pixels_2x][::16], 99.5)
    he_gray_2x = np.clip(he_stain_score_2x / max(stain_scale, 1e-6) * 255.0, 0, 255).astype(np.float32)
    he_rgb_display = np.stack([
        np.clip(channel / white_reference, 0.0, 1.0)
        for channel, white_reference in zip(he_channels_2x[:3], he_white_references)
    ], axis=-1)
elif he_channels_2x:
    he_gray_2x = clamp_and_normalize(he_channels_2x[0], CLAMP_SIGMA)

if channel_positive_count_2x is not None:
    print(f"  Channel-wise artifact maps computed from {len(codex_indices)} CODEX channels "
          f"(robust-Z breadth threshold={BRIGHT_BREADTH_CHANNEL_SIGMA:.1f}).")

codex_mask_2x = codex_proj_2x >= TISSUE_THRESHOLD
if he_stain_score_2x is not None:
    valid_stain_values = he_stain_score_2x[he_valid_pixels_2x]
    otsu_val_2x = threshold_otsu(valid_stain_values[::16])
    he_od_threshold_2x = otsu_val_2x * HE_OD_THRESHOLD_FACTOR
    he_mask_2x = (he_stain_score_2x >= he_od_threshold_2x) & he_valid_pixels_2x

    if TISSUE_MASK_MODE == "he_only":
        tissue_mask_2d_2x = he_mask_2x.copy()
    elif TISSUE_MASK_MODE == "he_with_codex_edge_rescue":
        rescue_px = int(np.ceil(HE_CODEX_EDGE_RESCUE_MICRONS / (MICRONS_PER_PIXEL * BLUR_DOWNSAMPLE)))
        rescue_zone = cv2.dilate(he_mask_2x.astype(np.uint8), disk(max(rescue_px, 1)).astype(np.uint8)) > 0
        tissue_mask_2d_2x = he_mask_2x | (codex_mask_2x & rescue_zone)
    elif TISSUE_MASK_MODE == "he_codex_union":
        tissue_mask_2d_2x = he_mask_2x | codex_mask_2x
    else:
        raise ValueError(f"Unknown TISSUE_MASK_MODE: {TISSUE_MASK_MODE}")
else:
    otsu_val_2x = None
    he_od_threshold_2x = None
    he_mask_2x = None
    tissue_mask_2d_2x = codex_mask_2x
    print("  H&E channels not found; falling back to CODEX tissue masking.")

border_px = int(np.ceil(TISSUE_BORDER_EXCLUSION_UM / (MICRONS_PER_PIXEL * BLUR_DOWNSAMPLE)))
if border_px > 0:
    tissue_mask_2d_2x[:border_px, :] = False
    tissue_mask_2d_2x[-border_px:, :] = False
    tissue_mask_2d_2x[:, :border_px] = False
    tissue_mask_2d_2x[:, -border_px:] = False

morphology_kernel = disk(TISSUE_MORPH_RADIUS).astype(np.uint8)
tissue_u8 = tissue_mask_2d_2x.astype(np.uint8)
tissue_u8 = cv2.morphologyEx(tissue_u8, cv2.MORPH_CLOSE, morphology_kernel)
tissue_u8 = cv2.morphologyEx(tissue_u8, cv2.MORPH_OPEN, morphology_kernel)

component_count, component_labels, component_stats, _ = cv2.connectedComponentsWithStats(tissue_u8, connectivity=4)
if component_count > 1:
    component_sizes = component_stats[1:, cv2.CC_STAT_AREA]
    keep_components = np.zeros(component_count, dtype=bool)
    keep_components[np.where(component_sizes >= MIN_TISSUE_ISLAND_PX)[0] + 1] = True
    tissue_mask_2d_2x = keep_components[component_labels]
else:
    tissue_mask_2d_2x = tissue_u8 > 0

if border_px > 0:
    tissue_mask_2d_2x[:border_px, :] = False
    tissue_mask_2d_2x[-border_px:, :] = False
    tissue_mask_2d_2x[:, :border_px] = False
    tissue_mask_2d_2x[:, -border_px:] = False

t_proc = time.time()
print(f"  [Timing] Optical-density thresholding & morphology: {t_proc - t_io:.1f}s")

pct_tissue = 100.0 * tissue_mask_2d_2x.mean()
print(f"Step 1.2 tissue mask complete ({TISSUE_MASK_MODE}, {BLUR_DOWNSAMPLE}x). Coverage: {pct_tissue:.1f}%")
if he_od_threshold_2x is not None:
    print(f"  H&E white references: {[round(value, 1) for value in he_white_references]}")
    print(f"  H&E OD threshold: {he_od_threshold_2x:.4f} (Otsu={otsu_val_2x:.4f}, factor={HE_OD_THRESHOLD_FACTOR})")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), dpi=120, constrained_layout=True)
fig.suptitle("H&E Optical-Density Tissue Masking", fontsize=15, fontweight='bold')

if he_stain_score_2x is not None:
    histogram_values = he_stain_score_2x[he_valid_pixels_2x][::16]
    axes[0].hist(histogram_values, bins=200, color='#888888', alpha=0.75, edgecolor='none')
    axes[0].axvline(he_od_threshold_2x, color='red', linestyle='--', linewidth=2,
                    label=f'OD threshold ({he_od_threshold_2x:.3f})')
    axes[0].set_xlabel("Mean H&E optical density")
else:
    axes[0].hist(codex_proj_2x[::8, ::8].ravel(), bins=200, color='#888888', alpha=0.75, edgecolor='none')
    axes[0].set_xlabel("CODEX projection intensity")
axes[0].set_title("Background vs. Stain Distribution", fontweight='bold')
axes[0].set_ylabel("Pixel Count")
axes[0].legend(loc='upper right')

if he_rgb_display is not None:
    axes[1].imshow(he_rgb_display, aspect='equal')
elif he_gray_2x is not None:
    axes[1].imshow(he_gray_2x, cmap='gray', aspect='equal')
else:
    axes[1].imshow(codex_proj_2x, cmap='gray', aspect='equal')
axes[1].set_title(f"Virtual H&E ({BLUR_DOWNSAMPLE}x downsampled)", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(tissue_mask_2d_2x, cmap='gray', aspect='equal')
axes[2].set_title(f"{TISSUE_MASK_MODE} Mask ({pct_tissue:.1f}% tissue)", fontweight='bold')
axes[2].axis('off')
plt.show()

print("\nGenerating H&E tissue boundary GeoJSON...")
tissue_u8 = tissue_mask_2d_2x.astype(np.uint8) * 255
contours_cv, hierarchy_cv = cv2.findContours(tissue_u8, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

boundary_features = []
if hierarchy_cv is not None:
    hierarchy = hierarchy_cv[0]
    contour_index = 0
    while contour_index >= 0:
        exterior = cv2.approxPolyDP(contours_cv[contour_index], 2.0, closed=True).reshape(-1, 2)
        if len(exterior) >= 3:
            exterior_coords = [(float(point[0]) * BLUR_DOWNSAMPLE, float(point[1]) * BLUR_DOWNSAMPLE) for point in exterior]
            holes = []
            child = hierarchy[contour_index][2]
            while child >= 0:
                hole = cv2.approxPolyDP(contours_cv[child], 2.0, closed=True).reshape(-1, 2)
                if len(hole) >= 3:
                    holes.append([(float(point[0]) * BLUR_DOWNSAMPLE, float(point[1]) * BLUR_DOWNSAMPLE) for point in hole])
                child = hierarchy[child][0]
            try:
                polygon = make_valid(Polygon(exterior_coords, holes))
                if not polygon.is_empty:
                    boundary_features.append({
                        "type": "Feature", "id": str(uuid.uuid4()),
                        "geometry": mapping(polygon),
                        "properties": {
                            "objectType": "annotation",
                            "classification": {"name": "Tissue_Boundary", "colorRGB": -16711936},
                            "isLocked": True
                        }
                    })
            except Exception:
                pass
        contour_index = hierarchy[contour_index][0]

tissue_boundary_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_tissue_boundary.geojson")
with open(tissue_boundary_path, "w") as file_handle:
    json.dump({"type": "FeatureCollection", "features": boundary_features}, file_handle, separators=(",", ":"))

t_end = time.time()
print(f"  TISSUE BOUNDARY GEOJSON EXPORTED: {tissue_boundary_path}")
print(f"  [Timing] TOTAL CELL 1.2: {t_end - t_s12_start:.1f}s")

### Step 1.2b — Blur, Bright Artifact, and External-Seed Filamentous Lint Detection

**Objective:** Detect three independent artifact classes while preserving true tissue:

1. **Out-of-focus tissue** is identified from normalized Laplacian focus loss relative to the virtual H&E optical-density reference.
2. **Flat bright artifacts** are identified conservatively inside the H&E tissue mask.
3. **Filamentous lint** is seeded only in H&E-negative space from the strongest robust signal in any individual CODEX channel, then filtered by physical length, width, aspect ratio, filamentarity, and seam geometry.

The lint model does not require co-localization across acquisition cycles, so single-marker or shifted fibers remain detectable. Thin repeated horizontal tile/scan seams are rejected explicitly. Accepted lint is exported to `*_detected_lint.geojson`; only overlap with H&E-defined tissue is subtracted from `*_valid_tissue.geojson`.

**Precision-first limitation:** a fiber entirely enclosed within H&E-positive tissue and lacking an external H&E-negative segment is not automatically removed, because its shape alone is not sufficient to distinguish it safely from vessels, glands, or stromal tracks. Such cases should be reviewed in QuPath using the separate lint GeoJSON.

In [ ]:
# ============================================================================
# STEP 1.2b: Blurry Region, Bright Artifact, and External-Seed Lint Detection
# ============================================================================
# Blur is evaluated only in H&E-defined tissue. Filamentous lint is seeded outside
# tissue to avoid classifying marker-positive glands, vessels, or stromal tracks as lint.

import time
from skimage.morphology import skeletonize

t_cell_start = time.time()

# --- Blur detection settings ---
USE_HE_REFERENCE       = True
BLUR_RATIO_THRESHOLD   = 0.35
HE_STRUCTURE_FRACTION  = 0.02
LAPLACIAN_KSIZE        = 5
WINDOW_SIZE            = 31
GAUSS_PRE_SIGMA        = 1.5
GAUSS_POST_SIGMA       = 5.0
MORPH_CLOSE_RADIUS     = 25
MORPH_OPEN_RADIUS      = 7
EXPAND_MICRONS         = 35.0
MIN_AREA_MICRONS2      = 1000.0

# --- Conservative non-filament bright artifact settings ---
DETECT_BRIGHT_ARTIFACTS        = True
USE_BREADTH_SIGNAL             = False
BRIGHT_BREADTH_FRACTION        = 0.50
BRIGHT_FLAT_MEAN_PERCENTILE    = 99.5
BRIGHT_FLAT_TEXTURE_PERCENTILE = 2.0
BRIGHT_MORPH_CLOSE_RADIUS      = 9
BRIGHT_MORPH_OPEN_RADIUS       = 2
BRIGHT_EXPAND_MICRONS          = 10.0
MIN_AREA_MICRONS2_BRIGHT       = 20.0

# --- Filamentous lint settings ---
# P28 calibration: Z>=12 plus a 5-um minimum width retained long external fibers
# while eliminating the repeated thin horizontal scan/tile seams.
DETECT_FILAMENTOUS_LINT      = True
LINT_ROBUST_Z_THRESHOLD      = 12.0
LINT_HE_EXCLUSION_MARGIN_UM  = 5.0
LINT_MIN_AREA_MICRONS2       = 20.0
LINT_MAX_AREA_MICRONS2       = 100000.0
LINT_MIN_LENGTH_MICRONS      = 75.0
LINT_MIN_WIDTH_MICRONS       = 5.0
LINT_MAX_WIDTH_MICRONS       = 60.0
LINT_MIN_ASPECT_RATIO        = 4.0
LINT_MORPH_CLOSE_RADIUS      = 3
LINT_MORPH_OPEN_RADIUS       = 0
LINT_EXPAND_MICRONS          = 5.0
LINT_SEAM_MAX_WIDTH_MICRONS  = 8.0
LINT_SEAM_MIN_SPAN_RATIO     = 5.0

def _gauss_ksize(sigma):
    radius = int(4 * sigma + 0.5)
    return (2 * radius + 1, 2 * radius + 1)

def _remove_small_components(mask, min_area_px, connectivity=4):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=connectivity)
    if count <= 1:
        return np.zeros_like(mask, dtype=bool)
    keep = np.zeros(count, dtype=bool)
    keep[np.where(stats[1:, cv2.CC_STAT_AREA] >= min_area_px)[0] + 1] = True
    return keep[labels]

def _mask_to_polygons(mask, scale_factor, simplify_px=2.0):
    contours, hierarchy = cv2.findContours(mask.astype(np.uint8) * 255, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    if hierarchy is None:
        return polygons
    hierarchy = hierarchy[0]
    contour_index = 0
    while contour_index >= 0:
        exterior = cv2.approxPolyDP(contours[contour_index], simplify_px, closed=True).reshape(-1, 2)
        if len(exterior) >= 3:
            exterior_coords = [(float(point[0]) * scale_factor, float(point[1]) * scale_factor) for point in exterior]
            holes = []
            child = hierarchy[contour_index][2]
            while child >= 0:
                hole = cv2.approxPolyDP(contours[child], simplify_px, closed=True).reshape(-1, 2)
                if len(hole) >= 3:
                    holes.append([(float(point[0]) * scale_factor, float(point[1]) * scale_factor) for point in hole])
                child = hierarchy[child][0]
            try:
                polygon = make_valid(Polygon(exterior_coords, holes))
                if not polygon.is_empty:
                    polygons.append(polygon)
            except Exception:
                pass
        contour_index = hierarchy[contour_index][0]
    return polygons

_KSIZE_PRE = _gauss_ksize(GAUSS_PRE_SIGMA)
_KSIZE_POST = _gauss_ksize(GAUSS_POST_SIGMA)
_WK = (WINDOW_SIZE, WINDOW_SIZE)
pixel_size_2x_um = MICRONS_PER_PIXEL * BLUR_DOWNSAMPLE
pixel_area_2x_um2 = pixel_size_2x_um ** 2
px_per_um_2x = 1.0 / pixel_size_2x_um

# --- Reload the Step 1.2 boundary so QuPath edits are respected ---
H_2x, W_2x = codex_proj_2x.shape
print(f"Reloading tissue mask from: {tissue_boundary_path}")
with open(tissue_boundary_path, 'r') as file_handle:
    boundary_geojson = json.load(file_handle)

boundary_polygons = []
for feature in boundary_geojson.get('features', []):
    geometry = feature.get('geometry')
    if geometry:
        try:
            polygon = shape(geometry)
            if not polygon.is_empty:
                boundary_polygons.append(polygon)
        except Exception:
            pass

if boundary_polygons:
    shapes_to_rasterize = []
    merged_boundary = unary_union(boundary_polygons)
    boundary_geometries = [merged_boundary] if merged_boundary.geom_type == 'Polygon' else list(merged_boundary.geoms)
    for polygon in boundary_geometries:
        exterior = [(x / BLUR_DOWNSAMPLE, y / BLUR_DOWNSAMPLE) for x, y in polygon.exterior.coords]
        interiors = [[(x / BLUR_DOWNSAMPLE, y / BLUR_DOWNSAMPLE) for x, y in ring.coords] for ring in polygon.interiors]
        try:
            shapes_to_rasterize.append((Polygon(exterior, interiors), 1))
        except Exception:
            pass
    tissue_mask_2d_2x = rasterio.features.rasterize(
        shapes_to_rasterize, out_shape=(H_2x, W_2x), fill=0, default_value=1, dtype=np.uint8
    ).astype(bool)
    print(f"  Tissue mask reloaded ({tissue_mask_2d_2x.sum():,} px, {100.0 * tissue_mask_2d_2x.mean():.1f}% coverage)")
else:
    print("  No polygons found; using the in-memory Step 1.2 mask.")

t_reload = time.time()
print(f"  [Timing] Tissue mask reload: {t_reload - t_cell_start:.1f}s")

# --- Normalized Laplacian focus scoring ---
def compute_normalized_lap_var(img_f32):
    smoothed = cv2.GaussianBlur(img_f32, _KSIZE_PRE, GAUSS_PRE_SIGMA, borderType=cv2.BORDER_REFLECT)
    laplacian = cv2.Laplacian(smoothed, cv2.CV_32F, ksize=LAPLACIAN_KSIZE)
    mean_lap_sq = cv2.boxFilter(laplacian * laplacian, ddepth=-1, ksize=_WK, borderType=cv2.BORDER_REFLECT)
    mean_lap = cv2.boxFilter(laplacian, ddepth=-1, ksize=_WK, borderType=cv2.BORDER_REFLECT)
    laplacian_variance = np.maximum(mean_lap_sq - mean_lap * mean_lap, 0.0)
    local_mean = cv2.boxFilter(img_f32, ddepth=-1, ksize=_WK, borderType=cv2.BORDER_REFLECT)
    normalized_variance = laplacian_variance / (local_mean * local_mean + 1.0)
    return cv2.GaussianBlur(normalized_variance, _KSIZE_POST, GAUSS_POST_SIGMA, borderType=cv2.BORDER_REFLECT)

codex_focus_2x = compute_normalized_lap_var(codex_proj_2x)
he_focus_2x = compute_normalized_lap_var(he_gray_2x) if he_gray_2x is not None else None
t_focus = time.time()
print(f"  [Timing] CODEX/H&E focus scoring: {t_focus - t_reload:.1f}s")

if he_focus_2x is not None and USE_HE_REFERENCE:
    ratio_map_smooth = cv2.GaussianBlur(
        codex_focus_2x / (he_focus_2x + 1e-6), _KSIZE_POST, GAUSS_POST_SIGMA,
        borderType=cv2.BORDER_REFLECT
    )
    he_values = he_focus_2x[tissue_mask_2d_2x]
    he_mean = he_values.mean() if he_values.size else 1.0
    he_has_structure = he_focus_2x >= he_mean * HE_STRUCTURE_FRACTION
    blur_mask_2x = (ratio_map_smooth < BLUR_RATIO_THRESHOLD) & tissue_mask_2d_2x & he_has_structure
else:
    tissue_focus_values = codex_focus_2x[tissue_mask_2d_2x]
    focus_threshold = np.percentile(tissue_focus_values, 20.0) if tissue_focus_values.size else -np.inf
    blur_mask_2x = (codex_focus_2x <= focus_threshold) & tissue_mask_2d_2x

blur_refined_u8 = blur_mask_2x.astype(np.uint8)
if MORPH_CLOSE_RADIUS > 0:
    blur_refined_u8 = cv2.morphologyEx(blur_refined_u8, cv2.MORPH_CLOSE, disk(MORPH_CLOSE_RADIUS).astype(np.uint8))
if MORPH_OPEN_RADIUS > 0:
    blur_refined_u8 = cv2.morphologyEx(blur_refined_u8, cv2.MORPH_OPEN, disk(MORPH_OPEN_RADIUS).astype(np.uint8))
blur_refined_2x = (blur_refined_u8 > 0) & tissue_mask_2d_2x

if EXPAND_MICRONS > 0:
    expand_px = int(np.ceil(EXPAND_MICRONS * px_per_um_2x))
    blur_refined_2x = (cv2.dilate(blur_refined_2x.astype(np.uint8), disk(expand_px).astype(np.uint8)) > 0) & tissue_mask_2d_2x

min_blur_area_px = int(np.ceil(MIN_AREA_MICRONS2 / pixel_area_2x_um2))
blur_refined_2x = _remove_small_components(blur_refined_2x, min_blur_area_px, connectivity=4)
t_blur = time.time()
print(f"  [Timing] Blur detection/refinement: {t_blur - t_focus:.1f}s")

# --- Non-filament bright artifacts inside tissue ---
if DETECT_BRIGHT_ARTIFACTS:
    if USE_BREADTH_SIGNAL and channel_positive_count_2x is not None and len(codex_indices) > 0:
        breadth_fraction = channel_positive_count_2x.astype(np.float32) / float(len(codex_indices))
        breadth_mask = (breadth_fraction >= BRIGHT_BREADTH_FRACTION) & tissue_mask_2d_2x
    else:
        breadth_mask = np.zeros_like(tissue_mask_2d_2x, dtype=bool)

    local_mean_codex = cv2.boxFilter(codex_proj_2x, ddepth=-1, ksize=_WK, borderType=cv2.BORDER_REFLECT)
    mean_values = local_mean_codex[tissue_mask_2d_2x]
    focus_values = codex_focus_2x[tissue_mask_2d_2x]
    bright_mean_threshold = np.percentile(mean_values, BRIGHT_FLAT_MEAN_PERCENTILE) if mean_values.size else np.inf
    flat_texture_threshold = np.percentile(focus_values, BRIGHT_FLAT_TEXTURE_PERCENTILE) if focus_values.size else -np.inf
    flat_glare_mask = (
        (local_mean_codex >= bright_mean_threshold)
        & (codex_focus_2x <= flat_texture_threshold)
        & tissue_mask_2d_2x
    )
    bright_u8 = (breadth_mask | flat_glare_mask).astype(np.uint8)
    if BRIGHT_MORPH_CLOSE_RADIUS > 0:
        bright_u8 = cv2.morphologyEx(bright_u8, cv2.MORPH_CLOSE, disk(BRIGHT_MORPH_CLOSE_RADIUS).astype(np.uint8))
    if BRIGHT_MORPH_OPEN_RADIUS > 0:
        bright_u8 = cv2.morphologyEx(bright_u8, cv2.MORPH_OPEN, disk(BRIGHT_MORPH_OPEN_RADIUS).astype(np.uint8))
    nonfilament_bright_2x = (bright_u8 > 0) & tissue_mask_2d_2x
    if BRIGHT_EXPAND_MICRONS > 0:
        expand_px_bright = int(np.ceil(BRIGHT_EXPAND_MICRONS * px_per_um_2x))
        nonfilament_bright_2x = (
            cv2.dilate(nonfilament_bright_2x.astype(np.uint8), disk(expand_px_bright).astype(np.uint8)) > 0
        ) & tissue_mask_2d_2x
    min_bright_area_px = int(np.ceil(MIN_AREA_MICRONS2_BRIGHT / pixel_area_2x_um2))
    nonfilament_bright_2x = _remove_small_components(nonfilament_bright_2x, min_bright_area_px, connectivity=4)
else:
    nonfilament_bright_2x = np.zeros_like(tissue_mask_2d_2x, dtype=bool)

# --- H&E-external filamentous lint detection ---
lint_component_records = []
lint_mask_2x = np.zeros_like(tissue_mask_2d_2x, dtype=bool)
lint_candidate_2x = np.zeros_like(tissue_mask_2d_2x, dtype=bool)

if DETECT_FILAMENTOUS_LINT:
    if 'codex_robust_max_2x' not in globals() or codex_robust_max_2x is None:
        print("  codex_robust_max_2x is unavailable; rerun Step 1.2 to enable filament detection.")
    else:
        exclusion_px = int(np.ceil(LINT_HE_EXCLUSION_MARGIN_UM * px_per_um_2x))
        tissue_exclusion = cv2.dilate(
            tissue_mask_2d_2x.astype(np.uint8), disk(max(exclusion_px, 1)).astype(np.uint8)
        ) > 0
        lint_search_region_2x = ~tissue_exclusion
        lint_candidate_2x = (codex_robust_max_2x >= LINT_ROBUST_Z_THRESHOLD) & lint_search_region_2x

        search_border_px = globals().get('border_px', 0)
        if search_border_px > 0:
            lint_candidate_2x[:search_border_px, :] = False
            lint_candidate_2x[-search_border_px:, :] = False
            lint_candidate_2x[:, :search_border_px] = False
            lint_candidate_2x[:, -search_border_px:] = False

        lint_candidate_u8 = lint_candidate_2x.astype(np.uint8)
        if LINT_MORPH_CLOSE_RADIUS > 0:
            lint_candidate_u8 = cv2.morphologyEx(
                lint_candidate_u8, cv2.MORPH_CLOSE, disk(LINT_MORPH_CLOSE_RADIUS).astype(np.uint8)
            )
        if LINT_MORPH_OPEN_RADIUS > 0:
            lint_candidate_u8 = cv2.morphologyEx(
                lint_candidate_u8, cv2.MORPH_OPEN, disk(LINT_MORPH_OPEN_RADIUS).astype(np.uint8)
            )

        component_count, component_labels, component_stats, _ = cv2.connectedComponentsWithStats(
            lint_candidate_u8, connectivity=8
        )
        for component_id in range(1, component_count):
            x = component_stats[component_id, cv2.CC_STAT_LEFT]
            y = component_stats[component_id, cv2.CC_STAT_TOP]
            width = component_stats[component_id, cv2.CC_STAT_WIDTH]
            height = component_stats[component_id, cv2.CC_STAT_HEIGHT]
            area_px = component_stats[component_id, cv2.CC_STAT_AREA]
            area_um2 = area_px * pixel_area_2x_um2
            if area_um2 < LINT_MIN_AREA_MICRONS2 or area_um2 > LINT_MAX_AREA_MICRONS2:
                continue

            component_roi = component_labels[y:y + height, x:x + width] == component_id
            contours, _ = cv2.findContours(component_roi.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            if not contours:
                continue
            contour = max(contours, key=cv2.contourArea)
            rect_width, rect_height = cv2.minAreaRect(contour)[1]
            major_axis_px = max(rect_width, rect_height, 1.0)
            minor_axis_px = max(min(rect_width, rect_height), 1.0)
            rotated_aspect = major_axis_px / minor_axis_px

            skeleton_length_px = float(skeletonize(component_roi).sum())
            mean_width_px = area_px / max(skeleton_length_px, 1.0)
            estimated_length_um = max(major_axis_px, skeleton_length_px) * pixel_size_2x_um
            estimated_width_um = mean_width_px * pixel_size_2x_um
            filamentarity = skeleton_length_px / max(mean_width_px, 1.0)
            horizontal_span_ratio = width / max(height, 1)
            seam_like = (
                horizontal_span_ratio >= LINT_SEAM_MIN_SPAN_RATIO
                and estimated_width_um <= LINT_SEAM_MAX_WIDTH_MICRONS
            )

            accepted = (
                estimated_length_um >= LINT_MIN_LENGTH_MICRONS
                and LINT_MIN_WIDTH_MICRONS <= estimated_width_um <= LINT_MAX_WIDTH_MICRONS
                and rotated_aspect >= LINT_MIN_ASPECT_RATIO
                and filamentarity >= LINT_MIN_ASPECT_RATIO
                and not seam_like
            )

            lint_component_records.append({
                'component': component_id,
                'area_um2': area_um2,
                'length_um': estimated_length_um,
                'width_um': estimated_width_um,
                'aspect_ratio': rotated_aspect,
                'filamentarity': filamentarity,
                'horizontal_span_ratio': horizontal_span_ratio,
                'seam_like': seam_like,
                'accepted': accepted,
            })
            if accepted:
                lint_mask_2x[y:y + height, x:x + width][component_roi] = True

        if LINT_EXPAND_MICRONS > 0 and np.any(lint_mask_2x):
            lint_expand_px = int(np.ceil(LINT_EXPAND_MICRONS * px_per_um_2x))
            lint_mask_2x = cv2.dilate(
                lint_mask_2x.astype(np.uint8), disk(lint_expand_px).astype(np.uint8)
            ) > 0

lint_components_df = pd.DataFrame(lint_component_records)
if not lint_components_df.empty:
    lint_components_df = lint_components_df.sort_values(['accepted', 'length_um'], ascending=[False, False])
    accepted_count = int(lint_components_df['accepted'].sum())
    seam_count = int(lint_components_df['seam_like'].sum())
    print(f"  External filament candidates measured: {len(lint_components_df):,}; "
          f"accepted lint: {accepted_count:,}; rejected seam-like: {seam_count:,}")
    display(lint_components_df.head(25))
else:
    print("  No external filament candidates passed the minimum area filter.")

t_lint = time.time()
print(f"  [Timing] Bright/filament detection: {t_lint - t_blur:.1f}s")

# Only expanded lint overlapping tissue is subtracted; all lint is exported separately.
lint_in_tissue_2x = lint_mask_2x & tissue_mask_2d_2x
bright_refined_2x = nonfilament_bright_2x | lint_in_tissue_2x
artifact_mask_2x = blur_refined_2x | bright_refined_2x
valid_tissue_mask_2d_2x = tissue_mask_2d_2x & (~artifact_mask_2x)

tissue_pixels = max(int(tissue_mask_2d_2x.sum()), 1)
blurry_px = int(blur_refined_2x.sum())
bright_px = int(nonfilament_bright_2x.sum())
lint_total_px = int(lint_mask_2x.sum())
lint_tissue_px = int(lint_in_tissue_2x.sum())
artifact_px = int(artifact_mask_2x.sum())
valid_px = int(valid_tissue_mask_2d_2x.sum())
blurry_pct = 100.0 * blurry_px / tissue_pixels
bright_pct = 100.0 * bright_px / tissue_pixels
lint_tissue_pct = 100.0 * lint_tissue_px / tissue_pixels
artifact_pct = 100.0 * artifact_px / tissue_pixels
valid_pct = 100.0 * valid_px / tissue_pixels

print(f"  Blurry area                  : {blurry_pct:.2f}% of tissue ({blurry_px:,} px)")
print(f"  Non-filament bright area     : {bright_pct:.2f}% of tissue ({bright_px:,} px)")
print(f"  Filamentous lint             : {lint_total_px:,} px total; {lint_tissue_pct:.2f}% overlaps tissue")
print(f"  Combined artifact area       : {artifact_pct:.2f}% of tissue ({artifact_px:,} px)")
print(f"  Clean valid tissue retained  : {valid_pct:.2f}% ({valid_px:,} px)")

# --- Export valid tissue and detected lint as separate QuPath GeoJSON files ---
valid_polygons = _mask_to_polygons(valid_tissue_mask_2d_2x, float(BLUR_DOWNSAMPLE), simplify_px=2.0)
valid_geometry = make_valid(unary_union(valid_polygons)) if valid_polygons else None
valid_geojson_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_valid_tissue.geojson")
valid_features = []
if valid_geometry is not None and not valid_geometry.is_empty:
    valid_features.append({
        "type": "Feature", "id": str(uuid.uuid4()),
        "geometry": mapping(valid_geometry),
        "properties": {
            "objectType": "annotation",
            "classification": {"name": "Valid_Tissue", "colorRGB": -16711936},
            "name": "Valid_Tissue_Clean", "isLocked": False
        }
    })
with open(valid_geojson_path, "w") as file_handle:
    json.dump({"type": "FeatureCollection", "features": valid_features}, file_handle, indent=2)

lint_polygons = _mask_to_polygons(lint_mask_2x, float(BLUR_DOWNSAMPLE), simplify_px=1.0)
lint_geojson_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_detected_lint.geojson")
lint_features = []
if lint_polygons:
    lint_geometry = make_valid(unary_union(lint_polygons))
    if not lint_geometry.is_empty:
        lint_features.append({
            "type": "Feature", "id": str(uuid.uuid4()),
            "geometry": mapping(lint_geometry),
            "properties": {
                "objectType": "annotation",
                "classification": {"name": "Detected_Lint", "colorRGB": -16711681},
                "name": "Detected_Filamentous_Lint", "isLocked": False
            }
        })
with open(lint_geojson_path, "w") as file_handle:
    json.dump({"type": "FeatureCollection", "features": lint_features}, file_handle, indent=2)

print(f"  VALID TISSUE GEOJSON: {valid_geojson_path}")
print(f"  DETECTED LINT GEOJSON: {lint_geojson_path}")

# --- Diagnostics ---
fig, axes = plt.subplots(2, 2, figsize=(16, 13), dpi=120, constrained_layout=True)
fig.suptitle(f"{SAMPLE_ID} - Blur, Bright Artifact, and External-Seed Lint Diagnostics", fontsize=15, fontweight='bold')

axes[0, 0].imshow(tissue_mask_2d_2x, cmap='gray', aspect='equal')
axes[0, 0].set_title(f"H&E Tissue Mask ({100.0 * tissue_mask_2d_2x.mean():.1f}% image area)", fontweight='bold')
axes[0, 0].axis('off')

vmax = np.percentile(codex_robust_max_2x, 99.9) if codex_robust_max_2x is not None else 1.0
axes[0, 1].imshow(codex_robust_max_2x, cmap='magma', vmin=0, vmax=max(vmax, LINT_ROBUST_Z_THRESHOLD))
lint_contours = cv2.findContours(lint_mask_2x.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
for contour in lint_contours:
    contour_xy = contour.reshape(-1, 2)
    axes[0, 1].plot(contour_xy[:, 0], contour_xy[:, 1], color='#00ffff', linewidth=0.8)
axes[0, 1].set_title(f"External Robust Signal and Accepted Lint (cyan, Z >= {LINT_ROBUST_Z_THRESHOLD:g})", fontweight='bold')
axes[0, 1].axis('off')

background_gray = np.clip(255.0 - he_gray_2x, 0, 255).astype(np.uint8) if he_gray_2x is not None else codex_proj_2x.astype(np.uint8)
background_rgb = cv2.cvtColor(background_gray, cv2.COLOR_GRAY2RGB)
overlay = background_rgb.copy()
overlay[blur_refined_2x] = [255, 255, 0]
overlay[nonfilament_bright_2x] = [255, 0, 255]
overlay[lint_mask_2x] = [0, 255, 255]
blended = cv2.addWeighted(overlay, 0.55, background_rgb, 0.45, 0)
axes[1, 0].imshow(blended, aspect='equal')
axes[1, 0].set_title("Detected Artifacts", fontweight='bold')
axes[1, 0].axis('off')
axes[1, 0].legend(handles=[
    mpatches.Patch(color='#ffff00', label=f'Blurry ({blurry_pct:.2f}% tissue)'),
    mpatches.Patch(color='#ff00ff', label=f'Bright/flat ({bright_pct:.2f}% tissue)'),
    mpatches.Patch(color='#00ffff', label=f'External-seed lint ({lint_tissue_pct:.2f}% tissue overlap)'),
], loc='lower right', fontsize=8, framealpha=0.9)

axes[1, 1].imshow(valid_tissue_mask_2d_2x, cmap='gray', aspect='equal')
axes[1, 1].set_title(f"Clean Valid Tissue ({valid_pct:.2f}% retained)", fontweight='bold')
axes[1, 1].axis('off')
plt.show()

t_total = time.time()
print(f"\n  [Timing] TOTAL CELL 1.2b: {t_total - t_cell_start:.1f}s")

### Step 1.2c — QuPath Annotation Synchronization & Raster Scaling

**Objective:** Reload a pre-existing or user-edited tissue annotation GeoJSON (`s1r1_valid_tissue.geojson` or `s1r1_tissue_boundary.geojson`) and rasterize the binary mask to match the 4x clustering grid.

*Note: If you already have a custom tissue boundary GeoJSON drawn in QuPath, you can skip Steps 1.2 and 1.2b entirely and execute directly from this cell.*

In [ ]:
# ============================================================================
# STEP 1.2c: Sync QuPath GeoJSON & Scale to Clustering Downsample (4x)
# ============================================================================

# --- Fallback Resolution: ensure variables exist if Steps 1.2 & 1.2b were skipped ---
if 'valid_geojson_path' not in globals():
    valid_geojson_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_valid_tissue.geojson")
    if not os.path.exists(valid_geojson_path):
        _alt_geojson = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_tissue_boundary.geojson")
        if os.path.exists(_alt_geojson):
            valid_geojson_path = _alt_geojson

if 'codex_indices' not in globals() or not codex_indices:
    _he_pats = ["he_", "he ", "h&e", "he_red", "he_green", "he_blue"]
    _he_idx = [i for i, n in enumerate(all_channel_names) if any(n.lower().startswith(p) or n.lower() == p.strip('_') for p in _he_pats)]
    codex_indices = [i for i, n in enumerate(all_channel_names) if i not in _he_idx]

# --- Read GeoJSON annotation file safely ---
polygons_loaded = []
if os.path.exists(valid_geojson_path):
    print(f"Reading & syncing GeoJSON file from disk: {valid_geojson_path} ...")
    with open(valid_geojson_path, 'r') as f:
        geojson_data = json.load(f)
    for feat in geojson_data.get('features', []):
        geom = feat.get('geometry')
        if geom:
            try:
                shp = shape(geom)
                if not shp.is_empty and shp.is_valid:
                    polygons_loaded.append(shp)
            except Exception:
                pass
else:
    print(f"⚠ GeoJSON file not found at {valid_geojson_path}. Will default to full image domain.")

SCALE_4X = float(CLUSTERING_DOWNSAMPLE)
merged_loaded = unary_union(polygons_loaded) if polygons_loaded else None

_first_ch_idx = codex_indices[0] if codex_indices else 0
with tifffile.TiffFile(tiff_path) as tif:
    codex_proj_4x = read_pyramid_level(tif, _first_ch_idx, CLUSTERING_DOWNSAMPLE)

H_4x, W_4x = codex_proj_4x.shape

if merged_loaded is not None:
    shapes_to_rasterize = []
    geoms_list = [merged_loaded] if merged_loaded.geom_type == 'Polygon' else list(merged_loaded.geoms)
    for poly in geoms_list:
        ext_coords = [(x / SCALE_4X, y / SCALE_4X) for x, y in poly.exterior.coords]
        int_coords = [[(x / SCALE_4X, y / SCALE_4X) for x, y in ring.coords] for ring in poly.interiors]
        try:
            scaled_poly = Polygon(ext_coords, int_coords)
            shapes_to_rasterize.append((scaled_poly, 1))
        except: pass

    if shapes_to_rasterize:
        valid_tissue_mask_2d = rasterio.features.rasterize(
            shapes_to_rasterize, out_shape=(H_4x, W_4x), fill=0, default_value=1, dtype=np.uint8
        ).astype(bool)
    else:
        valid_tissue_mask_2d = valid_tissue_mask_2d_2x[::2, ::2] if 'valid_tissue_mask_2d_2x' in globals() else np.ones((H_4x, W_4x), dtype=bool)
else:
    valid_tissue_mask_2d = valid_tissue_mask_2d_2x[::2, ::2] if 'valid_tissue_mask_2d_2x' in globals() else np.ones((H_4x, W_4x), dtype=bool)

valid_tissue_mask_flat = valid_tissue_mask_2d.flatten()
synced_px  = valid_tissue_mask_2d.sum()
synced_pct = 100.0 * synced_px / valid_tissue_mask_2d.size
print(f"  ✓ Active Valid Tissue Mask scaled to {CLUSTERING_DOWNSAMPLE}x for clustering!")
print(f"  Grid size : {H_4x} x {W_4x} px | Valid tissue coverage : {synced_pct:.1f}% ({synced_px:,} px)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
fig.suptitle(f"{SAMPLE_ID} — Active Valid Tissue Mask (Scaled to {CLUSTERING_DOWNSAMPLE}x for Clustering)", fontsize=14, fontweight='bold')

axes[0].imshow(codex_proj_4x, cmap='gray', aspect='equal')
axes[0].contour(valid_tissue_mask_2d.astype(np.float32), levels=[0.5], colors=['#2ecc71'], linewidths=1.2)
axes[0].set_title(f"Active Valid Tissue Outline at {CLUSTERING_DOWNSAMPLE}x (Green)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(valid_tissue_mask_2d, cmap='gray', aspect='equal')
axes[1].set_title(f"Active Valid Tissue Binary Mask ({synced_pct:.1f}% coverage)", fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Step 1.3 — Multichannel Image Feature Stack Construction (4x Downsample)

**Objective:** Build a clean multi-channel spatial image tensor at 4x downsampling (`CLUSTERING_DOWNSAMPLE`), filtering out non-marker channels specified in `EXCLUDE_CHANNELS`.

In [ ]:
# ============================================================================
# STEP 1.3: Spatial Downsampling & Multichannel Image Array Construction (4x)
# ============================================================================

print(f"Constructing downsampled multichannel array at {CLUSTERING_DOWNSAMPLE}x (excluding {EXCLUDE_CHANNELS}) ...")

keep_indices = [i for i, name in enumerate(all_channel_names) if name not in EXCLUDE_CHANNELS]
codex_pages  = []
keep_names   = []

with tifffile.TiffFile(tiff_path) as tif:
    for idx in keep_indices:
        ds = read_pyramid_level(tif, idx, CLUSTERING_DOWNSAMPLE)
        codex_pages.append(ds)
        keep_names.append(all_channel_names[idx])

img_ds = np.stack(codex_pages, axis=0)
n_keep, H, W = img_ds.shape

print(f"Multichannel array constructed.")
print(f"  Shape (C, H, W) : ({n_keep}, {H}, {W})")
print(f"  Total pixels    : {H * W:,}")
print(f"  Valid tissue px : {valid_tissue_mask_flat.sum():,} ({100.0 * valid_tissue_mask_flat.sum() / (H*W):.1f}%)")

### Step 1.4 — Dynamic Arcsinh Transformation, Robust Percentile Scaling & Spatial Smoothing

**Objective:** Normalize multi-channel signal intensities across tissue pixels:
1. **Dynamic Arcsinh:** Performs per-marker background noise standard deviation estimation ($\sigma_{\text{bg}}$) to set marker-specific cofactors $c_j = \max(5.0, k \cdot \sigma_{\text{bg}, j})$.
2. **Robust Percentile Scaling:** Clips upper signal outliers at the 99.9th percentile and normalizes to $[0, 1]$.
3. **Spatial Feature Smoothing:** Applies a light Gaussian spatial filter ($\sigma=0.8$) to reduce pixel-level shot noise.

In [ ]:
# ============================================================================
# STEP 1.4: Arcsinh Transformation, Percentile Scaling & Spatial Smoothing
# ============================================================================

img_arcsinh = np.zeros_like(img_ds)
cofactors_used = {}

if ARCSINH_MODE.lower() == "dynamic":
    print("Calculating per-marker dynamic arcsinh cofactors on valid tissue pixels...")
    for c in range(n_keep):
        marker_name = keep_names[c]
        raw_ch = img_ds[c]
        tissue_pixels = raw_ch[valid_tissue_mask_2d]
        if len(tissue_pixels) > 100:
            p75 = np.percentile(tissue_pixels, 75)
            bg_subset = tissue_pixels[tissue_pixels <= p75]
            if len(bg_subset) > 50:
                mad = np.median(np.abs(bg_subset - np.median(bg_subset)))
                sigma_bg = 1.4826 * mad
                c_dyn = max(float(ARCSINH_DYNAMIC_MIN), float(ARCSINH_DYNAMIC_K) * sigma_bg)
            else:
                c_dyn = float(ARCSINH_COFACTOR)
        else:
            c_dyn = float(ARCSINH_COFACTOR)
        cofactors_used[marker_name] = round(c_dyn, 2)
        img_arcsinh[c] = np.arcsinh(raw_ch / c_dyn)
    print("Dynamic cofactors computed:")
    for m_name, c_val in cofactors_used.items():
        print(f"  - {m_name}: cofactor = {c_val}")
else:
    print(f"Applying fixed arcsinh cofactor ({ARCSINH_COFACTOR}) across all channels...")
    for c in range(n_keep):
        cofactors_used[keep_names[c]] = float(ARCSINH_COFACTOR)
    img_arcsinh = np.arcsinh(img_ds / ARCSINH_COFACTOR)

# --- Per-channel Robust Percentile Scaling to [0, 1] ---
img_norm = np.zeros_like(img_arcsinh)
for c in range(n_keep):
    ch = img_arcsinh[c]
    ch_min = ch.min()
    ch_max = np.percentile(ch, PERCENTILE_THRESHOLD)
    if ch_max - ch_min > 1e-8:
        img_norm[c] = np.clip((ch - ch_min) / (ch_max - ch_min), 0.0, 1.0)
    else:
        img_norm[c] = 0.0

# --- Apply Spatial Feature Smoothing ---
if SIGMA_SPATIAL_SMOOTHING > 0:
    print(f"Applying spatial smoothing (sigma={SIGMA_SPATIAL_SMOOTHING}) to each channel...")
    for c in range(n_keep):
        img_norm[c] = gaussian(img_norm[c], sigma=SIGMA_SPATIAL_SMOOTHING, preserve_range=True)
    print("Spatial smoothing complete.")

print("Transformation & normalization complete.")

# --- Visualization: Intensity distributions ---
vis_markers = ["E-cadherin", "Pan-Cytokeratin", "Vimentin", "aSMA"]
visual_indices = [keep_names.index(m) for m in vis_markers if m in keep_names]
if len(visual_indices) < 4: visual_indices = list(range(min(4, n_keep)))

fig, axes = plt.subplots(2, len(visual_indices), figsize=(4 * len(visual_indices), 6), dpi=120)
fig.suptitle("Signal Intensity Distribution (Pre vs. Post Processing)", fontsize=14, fontweight='bold')

for col, idx in enumerate(visual_indices):
    m_name = keep_names[idx]
    c_val = cofactors_used.get(m_name, ARCSINH_COFACTOR)
    axes[0, col].hist(img_ds[idx].flatten(), bins=50, color='teal', alpha=0.7)
    axes[0, col].set_title(f"{m_name} Raw", fontweight='bold')
    axes[0, col].set_yscale('log')
    axes[1, col].hist(img_norm[idx].flatten(), bins=50, color='coral', alpha=0.7)
    axes[1, col].set_title(f"{m_name} (c={c_val})", fontweight='bold')
    axes[1, col].set_yscale('log')

plt.tight_layout()
plt.show()

### Step 1.5 — Z-Score Standardization & Marker Feature Matrix Preparation

**Objective:** Standardize normalized channel intensities ($Z = (X - \mu) / \sigma$) across valid tissue pixels. The number of top-enriched markers retained per pixel (Top-K) is auto-detected from the elbow of the mean sorted Z-score profile (`TOP_K_MODE="auto"` in Cell 3), with a manual override available (`TOP_K_MODE="manual"`, fixed `TOP_K_MARKERS`).

In [ ]:
# ============================================================================
# STEP 1.5: Z-Score Standardization & Feature Matrix Preparation
# ============================================================================

pixels = img_norm.reshape(n_keep, -1).T
valid_indices = np.where(valid_tissue_mask_flat)[0]
pixels_z_valid = np.zeros((len(valid_indices), n_keep), dtype=np.float32)
valid_pixels = pixels[valid_indices]

for c in range(n_keep):
    ch_vals = valid_pixels[:, c]
    mean, std = ch_vals.mean(), ch_vals.std()
    if std > 1e-8:
        pixels_z_valid[:, c] = (ch_vals - mean) / std

pixels_z = np.zeros_like(pixels)
pixels_z[valid_indices] = pixels_z_valid

sorted_idx = np.argsort(pixels_z_valid, axis=1)  # ascending; column -1 = strongest marker

# ============================================================================
# Auto-detect optimal TOP_K_MARKERS (elbow of the mean sorted Z-score profile)
# ============================================================================
# Rank every pixel's markers from strongest to weakest Z-score, then average that sorted profile
# across all valid tissue pixels. This gives a "scree curve": rank-1 markers are on average very
# enriched, and the curve decays toward 0 as rank increases. The elbow of that curve is the point
# where adding another marker to the Top-K mask stops buying meaningfully enriched signal and starts
# just admitting noise — i.e., a data-driven estimate of how many markers are typically co-enriched
# per pixel in THIS sample/panel, rather than a fixed guess.
def find_elbow_k(values, k_min, k_max):
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    k_min = max(1, min(k_min, n))
    k_max = max(k_min, min(k_max, n))
    xs = np.arange(1, n + 1, dtype=np.float64)
    x_norm = (xs - xs[0]) / (xs[-1] - xs[0] + 1e-12)
    y_norm = (values - values.min()) / (values.max() - values.min() + 1e-12)
    line_vec = np.array([x_norm[-1] - x_norm[0], y_norm[-1] - y_norm[0]])
    line_vec = line_vec / (np.linalg.norm(line_vec) + 1e-12)
    pts = np.stack([x_norm - x_norm[0], y_norm - y_norm[0]], axis=1)
    proj_len = pts @ line_vec
    proj_pts = np.outer(proj_len, line_vec)
    dist = np.linalg.norm(pts - proj_pts, axis=1)
    dist[: k_min - 1] = -np.inf
    dist[k_max:] = -np.inf
    return int(np.argmax(dist)) + 1

mean_sorted_profile = np.sort(pixels_z_valid, axis=1)[:, ::-1].mean(axis=0)  # rank-1 = strongest, on average
TOP_K_AUTO_DETECTED = find_elbow_k(mean_sorted_profile, TOP_K_AUTO_MIN, TOP_K_AUTO_MAX)

if USE_TOP_K_MASKING:
    if TOP_K_MODE == "auto":
        TOP_K_EFFECTIVE = TOP_K_AUTO_DETECTED
        print(f"  ✓ Auto-detected Top-K = {TOP_K_EFFECTIVE} (elbow of mean sorted Z-score profile; search range [{TOP_K_AUTO_MIN},{TOP_K_AUTO_MAX}]).")
        print(f"    Set TOP_K_MODE='manual' in Cell 3 to override with a fixed TOP_K_MARKERS={TOP_K_MARKERS} instead.")
    else:
        TOP_K_EFFECTIVE = TOP_K_MARKERS
        print(f"  Manual Top-K = {TOP_K_EFFECTIVE} (auto-detection disabled: TOP_K_MODE='manual'; data-driven estimate was {TOP_K_AUTO_DETECTED}).")
else:
    TOP_K_EFFECTIVE = TOP_K_MARKERS  # unused when masking is disabled, kept defined for downstream references

mask_valid = np.zeros_like(pixels_z_valid, dtype=bool)
top_k_idx  = sorted_idx[:, -TOP_K_EFFECTIVE:]
rows       = np.arange(pixels_z_valid.shape[0])[:, None]
mask_valid[rows, top_k_idx] = True

if USE_TOP_K_MASKING:
    X_valid = np.zeros_like(pixels_z_valid)
    X_valid[mask_valid] = pixels_z_valid[mask_valid]
    print(f"  ✓ Top-{TOP_K_EFFECTIVE} Feature Masking APPLIED: Only top {TOP_K_EFFECTIVE} marker Z-scores per pixel retained (non-top markers set to 0.0).")
else:
    X_valid = pixels_z_valid
    print("  ✓ Continuous Z-score features APPLIED across all channels (Top-K masking disabled).")

print(f"Valid tissue Z-score feature matrix ready ({X_valid.shape[0]:,} px x {X_valid.shape[1]} features).")

target_m1, target_m2 = "HLA-A", "Pan-Cytokeratin"
idx1 = keep_names.index(target_m1) if target_m1 in keep_names else 0
idx2 = keep_names.index(target_m2) if target_m2 in keep_names else min(1, n_keep-1)
name1, name2 = keep_names[idx1], keep_names[idx2]

fig, axes = plt.subplots(1, 3, figsize=(19, 5), dpi=120)
fig.suptitle("Z-Score Enrichment Effect & Auto Top-K Elbow Detection", fontsize=14, fontweight='bold')

axes[0].hist(pixels_z_valid[:, idx1], bins=50, color='blue', alpha=0.5, label=f'{name1} Z-score')
axes[0].hist(pixels_z_valid[:, idx2], bins=50, color='red', alpha=0.5, label=f'{name2} Z-score')
axes[0].set_title("Z-score distribution (Tissue Pixels)", fontweight='bold')
axes[0].set_xlabel("Z-score value")
axes[0].set_ylabel("Count")
axes[0].legend()

selected_counts = mask_valid.sum(axis=0)
selected_pcts   = 100.0 * selected_counts / len(valid_indices)

axes[1].bar([name1, name2], [selected_pcts[idx1], selected_pcts[idx2]], color=['blue', 'red'], alpha=0.7)
axes[1].set_title("Top-K Selection Percentage", fontweight='bold')
axes[1].set_ylabel("Percentage of Pixels selected (%)")

ranks = np.arange(1, n_keep + 1)
axes[2].plot(ranks, mean_sorted_profile, color='#444444', linewidth=2, marker='o', markersize=3)
axes[2].axvline(TOP_K_EFFECTIVE, color='crimson', linestyle='--', linewidth=2,
                 label=f"Selected K={TOP_K_EFFECTIVE} ({'auto' if (USE_TOP_K_MASKING and TOP_K_MODE=='auto') else 'manual'})")
if USE_TOP_K_MASKING and TOP_K_MODE != "auto":
    axes[2].axvline(TOP_K_AUTO_DETECTED, color='seagreen', linestyle=':', linewidth=2, label=f"Auto estimate K={TOP_K_AUTO_DETECTED}")
axes[2].set_title("Mean Sorted Z-score Profile (Elbow)", fontweight='bold')
axes[2].set_xlabel("Marker rank (1 = strongest per pixel)")
axes[2].set_ylabel("Mean Z-score across valid tissue")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


---
# Phase 2: Unsupervised Phenotype Discovery & Microenvironment Mapping

### Step 2.1 — MiniBatchKMeans Spatial Clustering & Distinct Visual Palette Assignment

**Objective:** Run MiniBatchKMeans unsupervised clustering ($k=20$) on standardized pixel feature vectors across valid tissue regions. Assigns a high-contrast qualitative color palette (`tab20` interleaved permutation) so adjacent cluster IDs are visually distinct across spatial maps.

In [ ]:
# ============================================================================
# STEP 2.1: MiniBatchKMeans Clustering & Automatic Color Assignment (Seed 42)
# ============================================================================

# --- Prepare K-Means feature matrix based on low-signal handling ---
if LOW_SIGNAL_HANDLING == "reserve":
    mean_signal_per_pixel = X_valid.mean(axis=1)
    low_signal_threshold = np.quantile(mean_signal_per_pixel, LOW_SIGNAL_QUANTILE)
    low_signal_mask = mean_signal_per_pixel <= low_signal_threshold
    kmeans_mask = ~low_signal_mask
    features_for_kmeans = X_valid[kmeans_mask]
    print(f"Low-signal reserve: {low_signal_mask.sum():,} pixels ({100.0*low_signal_mask.mean():.1f}%) set aside as 'Low_Signal_Tissue'.")
else:
    low_signal_mask = np.zeros(len(X_valid), dtype=bool)
    kmeans_mask = np.ones(len(X_valid), dtype=bool)
    features_for_kmeans = X_valid

print(f"Fitting MiniBatchKMeans (k={N_CLUSTERS}, seed={RANDOM_SEED}, batch_size={KMEANS_BATCH_SIZE}) on {features_for_kmeans.shape[0]:,} valid tissue pixels ...")
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=KMEANS_BATCH_SIZE,
    random_state=RANDOM_SEED,
    n_init=KMEANS_N_INIT,
    max_iter=KMEANS_MAX_ITER,
    verbose=0
)

labels_for_kmeans = kmeans.fit_predict(features_for_kmeans)

labels_valid = np.empty(len(X_valid), dtype=np.int32)
labels_valid[kmeans_mask] = labels_for_kmeans
if LOW_SIGNAL_HANDLING == "reserve":
    labels_valid[low_signal_mask] = N_CLUSTERS
    reserved_low_signal_cluster = N_CLUSTERS
else:
    reserved_low_signal_cluster = None

cluster_map_2d = np.full((H, W), -1, dtype=np.int32)
cluster_map_2d.flat[valid_indices] = labels_valid

np.random.seed(RANDOM_SEED)

# High-contrast qualitative palette: interleaves tab20 hues so consecutive cluster IDs are visually distinct
if N_CLUSTERS <= 20:
    base_palette = sns.color_palette("tab20", 20)
    interleave_order = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
    color_palette = [base_palette[i] for i in interleave_order[:N_CLUSTERS]]
else:
    color_palette = sns.color_palette("gist_ncar", N_CLUSTERS)

cluster_colors_hex = [mcolors.to_hex(c) for c in color_palette]
if reserved_low_signal_cluster is not None:
    cluster_colors_hex.append("#888888")

print("Clustering complete.")
unique, counts = np.unique(labels_valid, return_counts=True)
for u, cnt in zip(unique, counts):
    pct = 100.0 * cnt / len(labels_valid)
    lbl_str = "Low_Signal_Tissue" if u == reserved_low_signal_cluster else f"Cluster {u:2d}"
    hex_str = "#888888" if u == reserved_low_signal_cluster else cluster_colors_hex[u]
    print(f"  {lbl_str:20s} ({hex_str}) : {cnt:8,} px ({pct:5.1f}% of valid tissue)")

### Step 2.2 — Cluster Signature Heatmaps & Marker Enrichment Profiles

**Objective:** Characterize unsupervised clusters by computing mean normalized intensity and mean Z-score profiles. Visualizes hierarchical cluster relationships with annotated clustermaps and reports top dominant and enriched markers for each cluster.

In [ ]:
# ============================================================================
# STEP 2.2: Cluster Signature Heatmaps & Marker Enrichment Profiles
# ============================================================================

pixels_orig_valid = pixels[valid_indices]
cluster_profiles  = pd.DataFrame(pixels_orig_valid, columns=keep_names).assign(cluster=labels_valid).groupby('cluster').mean()
pixels_z_valid_df = pd.DataFrame(pixels_z_valid, columns=keep_names).assign(cluster=labels_valid).groupby('cluster').mean()

g = sns.clustermap(
    cluster_profiles,
    cmap='YlOrRd',
    figsize=(16, 7),
    row_cluster=True,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[f'Cluster {i}' for i in cluster_profiles.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g.ax_heatmap.set_xlabel('Marker', fontsize=13, fontweight='bold')
g.ax_heatmap.set_ylabel('Cluster ID', fontsize=13, fontweight='bold')
g.fig.suptitle('Cluster Absolute Signatures (Mean Normalized Intensity)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

g_z = sns.clustermap(
    pixels_z_valid_df,
    cmap='RdBu_r',
    center=0.0,
    figsize=(16, 7),
    row_cluster=True,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[f'Cluster {i}' for i in pixels_z_valid_df.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_z.ax_heatmap.set_xlabel('Marker', fontsize=13, fontweight='bold')
g_z.ax_heatmap.set_ylabel('Cluster ID', fontsize=13, fontweight='bold')
g_z.fig.suptitle('Cluster Unique/Enriched Signatures (Mean Z-score)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("\nTop-3 Dominant vs. Enriched Markers per Cluster:")
print("=" * 80)
for cid in cluster_profiles.index:
    row_int = cluster_profiles.loc[cid]
    top3_int = row_int.nlargest(3)
    int_str = ", ".join([f"{m} ({v:.2f})" for m, v in top3_int.items()])

    row_z = pixels_z_valid_df.loc[cid]
    top3_z = row_z.nlargest(3)
    z_str = ", ".join([f"{m} (+{v:.2f})" for m, v in top3_z.items()])

    print(f"  Cluster {cid:2d}:")
    print(f"    Dominant (Intensity): {int_str}")
    print(f"    Enriched (Z-score):   {z_str}")
    print("-" * 80)

In [ ]:
# ============================================================================
# STEP 2.2a: Size-Aware, Marker-Balanced Cluster Enrichment Heatmap
# ============================================================================
# This third view does not replace the original mean Z-score heatmap above.
# Colors are balanced across markers; annotations remain interpretable Z-scores.

SIZE_SHRINKAGE_PRIOR_FRACTION = 0.25
BALANCED_MARKER_QUANTILE = 0.90
BALANCED_MIN_Z_SCALE = 0.50
BALANCED_COLOR_LIMIT = 2.50

# Reconstruct the cluster mean Z matrix if this cell is rerun independently.
if 'pixels_z_valid_df' not in globals():
    pixels_z_valid_df = (
        pd.DataFrame(pixels_z_valid, columns=keep_names)
        .assign(cluster=labels_valid)
        .groupby('cluster')
        .mean()
    )

# Pixel count at the 4x grid is proportional to polygon area.
cluster_sizes = (
    pd.Series(labels_valid)
    .value_counts()
    .reindex(pixels_z_valid_df.index)
    .fillna(0)
    .astype(np.int64)
)
cluster_fractions = cluster_sizes / max(int(cluster_sizes.sum()), 1)

# Reliability shrinkage reduces unstable means from scarce clusters without
# allowing large clusters to dominate the phenotype score.
median_cluster_size = float(cluster_sizes[cluster_sizes > 0].median())
shrinkage_prior_pixels = max(
    1.0, median_cluster_size * SIZE_SHRINKAGE_PRIOR_FRACTION
)
cluster_reliability = cluster_sizes / (cluster_sizes + shrinkage_prior_pixels)

tissue_marker_center = (
    pixels_z_valid_df.mul(cluster_sizes, axis=0).sum(axis=0)
    / max(float(cluster_sizes.sum()), 1.0)
)
size_shrunk_z_profiles = tissue_marker_center + (
    pixels_z_valid_df.sub(tissue_marker_center, axis=1)
    .mul(cluster_reliability, axis=0)
)

# Normalize color contrast per marker using a guarded robust scale. The floor
# avoids amplifying markers whose between-cluster differences are only noise.
shrunk_deviation = size_shrunk_z_profiles.sub(tissue_marker_center, axis=1)
marker_color_scale = (
    shrunk_deviation.abs()
    .quantile(BALANCED_MARKER_QUANTILE, axis=0)
    .clip(lower=BALANCED_MIN_Z_SCALE)
)
balanced_cluster_scores = (
    shrunk_deviation.divide(marker_color_scale, axis=1)
    .clip(-BALANCED_COLOR_LIMIT, BALANCED_COLOR_LIMIT)
)

size_aware_labels = [
    f'Cluster {cluster_id}'
    for cluster_id in balanced_cluster_scores.index
]

# Color encodes the balanced score. The printed value in each tile is the
# size-shrunk mean Z-score, not the rescaled color score.
g_balanced = sns.clustermap(
    balanced_cluster_scores,
    cmap='RdBu_r',
    center=0.0,
    vmin=-BALANCED_COLOR_LIMIT,
    vmax=BALANCED_COLOR_LIMIT,
    figsize=(16, 7),
    row_cluster=True,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=size_aware_labels,
    xticklabels=True,
    annot=size_shrunk_z_profiles.to_numpy(),
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_balanced.ax_heatmap.set_xlabel('Marker', fontsize=13, fontweight='bold')
g_balanced.ax_heatmap.set_ylabel('Cluster ID', fontsize=13, fontweight='bold')
g_balanced.figure.suptitle('Cluster Size-Aware Balanced Signatures (Shrunk Mean Z-score)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

cluster_size_summary = pd.DataFrame({
    'pixels_4x': cluster_sizes,
    'tissue_percent': 100.0 * cluster_fractions,
    'reliability': cluster_reliability,
}).sort_values('tissue_percent', ascending=False)

print('\nSize-aware heatmap calibration:')
print(f'  Median cluster area          : {median_cluster_size:,.0f} pixels at 4x')
print(f'  Shrinkage prior              : {shrinkage_prior_pixels:,.0f} pixels')
print(f'  Marker color-scale quantile  : {BALANCED_MARKER_QUANTILE:.0%}')
print(f'  Minimum marker Z scale       : {BALANCED_MIN_Z_SCALE:.2f}')
display(cluster_size_summary)

print('\nTop-3 size-aware enriched markers per cluster:')
print('=' * 80)
for cluster_id in size_shrunk_z_profiles.index:
    top_markers = size_shrunk_z_profiles.loc[cluster_id].nlargest(3)
    marker_text = ', '.join(
        f'{marker} ({value:+.2f})' for marker, value in top_markers.items()
    )
    print(
        f'  Cluster {cluster_id:2d} '
        f'({100.0 * cluster_fractions.loc[cluster_id]:.2f}% tissue, '
        f'r={cluster_reliability.loc[cluster_id]:.2f}): {marker_text}'
    )

In [ ]:
# STEP 2.2b: High-resolution GeoJSON export settings
# This cell is intentionally lightweight and may be rerun without reclustering.
MACRO_EXPORT_MODE = "fine_gapless"
FINE_EXPORT_SMOOTH_SIGMA = 2.0
FINE_EXPORT_MAX_SIGMA = 6.0
GAPLESS_SHARED_BOUNDARIES = True

# Add cluster IDs identified as residual lint, for example [3, 7].
VALID_TISSUE_EXCLUDED_CLUSTERS = []
MERGED_VALID_CLOSE_UM = 4.0
MERGED_VALID_FILL_HOLES_UM2 = 100.0

print("High-resolution export settings loaded.")
print(f"  Mode: {MACRO_EXPORT_MODE}")
print(f"  Excluded clusters: {VALID_TISSUE_EXCLUDED_CLUSTERS or 'none'}")

### Step 2.3 — High-Resolution Gapless Cluster & Reviewed Valid-Tissue Export

**Objective:** Export cluster annotations at the native 4x clustering grid rather than the previous 4x4 macro-tile grid. Class scores are Gaussian-smoothed sequentially and resolved with winner-take-all assignment, so every valid-tissue pixel belongs to exactly one cluster while memory remains bounded. The resulting polygons have shared boundaries at approximately 2.03 µm resolution; independent per-class vector polishing is disabled in gapless mode because it moves neighboring edges separately and creates cracks or overlaps.

Set `VALID_TISSUE_EXCLUDED_CLUSTERS` in Cell 4 to cluster IDs identified as residual lint. This cell also exports a single `*_valid_tissue_from_clusters_highres.geojson` annotation from all remaining clusters. Narrow raster cracks and only very small holes are repaired according to `MERGED_VALID_CLOSE_UM` and `MERGED_VALID_FILL_HOLES_UM2`, while larger removed-lint regions remain excluded.

In [ ]:
# ============================================================================
# STEP 2.3: High-Resolution Gapless Cluster and Merged Valid-Tissue GeoJSON
# ============================================================================

from collections import defaultdict
from shapely.affinity import scale as scale_geometry

# Legacy coarse-map helper retained for Step 2.4 and optional coarse export.
def macro_class_map(fine_class_map, tissue_mask, class_names, tile_size):
    cropped_height = (fine_class_map.shape[0] // tile_size) * tile_size
    cropped_width = (fine_class_map.shape[1] // tile_size) * tile_size
    if cropped_height == 0 or cropped_width == 0:
        raise ValueError("Image smaller than macro tile size.")

    fine_crop = fine_class_map[:cropped_height, :cropped_width]
    tissue_crop = tissue_mask[:cropped_height, :cropped_width]
    coarse_shape = (cropped_height // tile_size, cropped_width // tile_size)
    tissue_fraction = tissue_crop.reshape(
        coarse_shape[0], tile_size, coarse_shape[1], tile_size
    ).mean(axis=(1, 3))
    coarse_tissue = tissue_fraction >= MACRO_MIN_TISSUE_FRACTION

    class_scores = np.stack([
        ((fine_crop == name) & tissue_crop).reshape(
            coarse_shape[0], tile_size, coarse_shape[1], tile_size
        ).sum(axis=(1, 3))
        for name in class_names
    ])
    coarse_map = np.full(coarse_shape, "Non-Tissue", dtype=object)
    winning_indices = np.argmax(class_scores, axis=0)
    for class_index, class_name in enumerate(class_names):
        coarse_map[coarse_tissue & (winning_indices == class_index)] = class_name

    if MACRO_SMOOTH_SIGMA > 0:
        scores = np.stack([
            cv2.GaussianBlur(
                (coarse_map == name).astype(np.float32), (0, 0), MACRO_SMOOTH_SIGMA,
                borderType=cv2.BORDER_REFLECT
            )
            for name in class_names
        ])
        smoothed_indices = np.argmax(scores, axis=0)
        for class_index, class_name in enumerate(class_names):
            coarse_map[coarse_tissue & (smoothed_indices == class_index)] = class_name

    return coarse_map, tile_size * CLUSTERING_DOWNSAMPLE

def raw_shapes(macro_map, class_names):
    for class_name in class_names:
        binary_mask = macro_map == class_name
        if not np.any(binary_mask):
            continue
        yield from (
            (class_name, geometry)
            for geometry, _ in rasterio.features.shapes(
                binary_mask.astype(np.uint8), mask=binary_mask, connectivity=8
            )
        )

def qupath_color_from_hex(hex_str):
    hex_clean = hex_str.lstrip('#')
    rgb_int = int(hex_clean, 16)
    return rgb_int - 0x1000000 if rgb_int >= 0x800000 else rgb_int

def polish_boundary(poly, simplify_um, smooth_um, microns_per_pixel, quad_segs=4):
    px_per_um_local = 1.0 / microns_per_pixel
    if simplify_um > 0:
        poly = poly.simplify(simplify_um * px_per_um_local, preserve_topology=True)
    if smooth_um > 0 and not poly.is_empty:
        radius = smooth_um * px_per_um_local
        try:
            rounded = poly.buffer(
                radius, join_style='round', quad_segs=quad_segs
            ).buffer(
                -radius, join_style='round', quad_segs=quad_segs
            )
            rounded = make_valid(rounded)
            if not rounded.is_empty:
                poly = rounded
        except Exception:
            pass
        if simplify_um > 0:
            poly = poly.simplify(simplify_um * px_per_um_local, preserve_topology=True)
    return poly

def smooth_fine_label_partition(label_map, tissue_mask, class_ids, sigma):
    """Winner-take-all smoothing with bounded RAM and complete tissue coverage."""
    if sigma <= 0:
        result = label_map.astype(np.int16, copy=True)
        result[~tissue_mask] = -1
        return result

    best_scores = np.full(label_map.shape, -np.inf, dtype=np.float32)
    result = np.full(label_map.shape, -1, dtype=np.int16)
    for class_id in class_ids:
        class_mask = (label_map == class_id).astype(np.float32)
        score = cv2.GaussianBlur(
            class_mask, (0, 0), sigma, borderType=cv2.BORDER_REFLECT
        )
        update = tissue_mask & (score > best_scores)
        best_scores[update] = score[update]
        result[update] = class_id
        del class_mask, score, update

    # Every valid pixel had an original label; restore any numerical edge cases.
    unresolved = tissue_mask & (result < 0)
    result[unresolved] = label_map[unresolved].astype(np.int16)
    result[~tissue_mask] = -1
    return result

def partition_shapes(label_map, class_name_by_id):
    """Polygonize the complete integer partition once so shared edges stay identical."""
    valid = label_map >= 0
    for geometry, value in rasterio.features.shapes(
        label_map.astype(np.int16), mask=valid, connectivity=8
    ):
        class_id = int(value)
        if class_id in class_name_by_id:
            yield class_id, class_name_by_id[class_id], geometry

def scaled_valid_geometry(geometry, raster_scale):
    polygon = scale_geometry(
        shape(geometry), xfact=raster_scale, yfact=raster_scale, origin=(0, 0)
    )
    return make_valid(polygon)

def export_feature_collection(features, output_path):
    with open(output_path, "w") as file_handle:
        json.dump(
            {"type": "FeatureCollection", "features": features},
            file_handle, separators=(",", ":")
        )
    return os.path.getsize(output_path) / (1024 ** 2)

print("Creating high-resolution cluster partition...")
active_class_ids = sorted(int(value) for value in np.unique(cluster_map_2d) if value >= 0)
class_name_by_id = {
    class_id: (
        "Low_Signal_Tissue"
        if reserved_low_signal_cluster is not None and class_id == reserved_low_signal_cluster
        else f"Cluster_{class_id}"
    )
    for class_id in active_class_ids
}

if MACRO_EXPORT_MODE == "fine_gapless":
    export_scale = CLUSTERING_DOWNSAMPLE
    smoothing_sigma = float(FINE_EXPORT_SMOOTH_SIGMA)
    while True:
        export_label_map = smooth_fine_label_partition(
            cluster_map_2d, valid_tissue_mask_2d, active_class_ids, smoothing_sigma
        )
        shape_records = list(partition_shapes(export_label_map, class_name_by_id))
        if len(shape_records) <= MAX_RAW_POLYGONS or smoothing_sigma >= FINE_EXPORT_MAX_SIGMA:
            break
        next_sigma = min(FINE_EXPORT_MAX_SIGMA, smoothing_sigma * 1.5)
        print(f"  {len(shape_records):,} fragments at sigma={smoothing_sigma:.2f}; retrying sigma={next_sigma:.2f}")
        smoothing_sigma = next_sigma

    export_resolution_um = export_scale * MICRONS_PER_PIXEL
    export_mode_label = f"fine gapless partition (sigma={smoothing_sigma:.2f})"
else:
    raw_class_map_2d = np.full((H, W), "Non-Tissue", dtype=object)
    for class_id, class_name in class_name_by_id.items():
        raw_class_map_2d[cluster_map_2d == class_id] = class_name

    tile_candidates = [MACRO_TILE_SIZE, MACRO_TILE_SIZE * 2, MACRO_TILE_SIZE * 4]
    for candidate_tile_size in tile_candidates:
        macro_raw_map_2d, export_scale = macro_class_map(
            raw_class_map_2d, valid_tissue_mask_2d,
            list(class_name_by_id.values()), candidate_tile_size
        )
        legacy_records = list(raw_shapes(macro_raw_map_2d, list(class_name_by_id.values())))
        if len(legacy_records) <= MAX_RAW_POLYGONS:
            break

    name_to_id = {name: class_id for class_id, name in class_name_by_id.items()}
    shape_records = [
        (name_to_id[class_name], class_name, geometry)
        for class_name, geometry in legacy_records
    ]
    export_label_map = np.full(macro_raw_map_2d.shape, -1, dtype=np.int16)
    for class_id, class_name in class_name_by_id.items():
        export_label_map[macro_raw_map_2d == class_name] = class_id
    export_resolution_um = export_scale * MICRONS_PER_PIXEL
    export_mode_label = f"legacy coarse partition (tile={candidate_tile_size})"

print(f"  Export mode       : {export_mode_label}")
print(f"  Raster resolution : {export_resolution_um:.3f} um/pixel")
print(f"  Raw polygons      : {len(shape_records):,}")

# Every fine-mode tissue pixel has exactly one class before vectorization.
if MACRO_EXPORT_MODE == "fine_gapless":
    uncovered_pixels = int(np.sum(valid_tissue_mask_2d & (export_label_map < 0)))
    outside_pixels = int(np.sum((export_label_map >= 0) & (~valid_tissue_mask_2d)))
    print(f"  Partition check   : uncovered={uncovered_pixels:,}, outside={outside_pixels:,}")
    if uncovered_pixels or outside_pixels:
        raise RuntimeError("Fine cluster partition is not gapless within valid tissue.")

polygons_by_class = defaultdict(list)
for class_id, class_name, geometry in shape_records:
    try:
        polygon = scaled_valid_geometry(geometry, export_scale)
        if not polygon.is_empty:
            polygons_by_class[class_id].append(polygon)
    except Exception:
        continue

raw_features = []
for class_id in active_class_ids:
    polygons = polygons_by_class.get(class_id, [])
    if not polygons:
        continue
    try:
        merged_geometry = make_valid(unary_union(polygons))
        if merged_geometry.is_empty:
            continue

        # Independent simplify/buffer operations move shared edges and create gaps.
        # Use them only for the legacy/non-gapless mode.
        if not GAPLESS_SHARED_BOUNDARIES:
            merged_geometry = polish_boundary(
                merged_geometry, BOUNDARY_SIMPLIFY_UM, BOUNDARY_SMOOTH_UM,
                MICRONS_PER_PIXEL, BOUNDARY_SMOOTH_QUAD_SEGS
            )
        if merged_geometry.is_empty:
            continue

        color_index = min(class_id, len(cluster_colors_hex) - 1)
        class_name = class_name_by_id[class_id]
        raw_features.append({
            "type": "Feature", "id": str(uuid.uuid4()),
            "geometry": mapping(merged_geometry),
            "properties": {
                "objectType": "annotation",
                "classification": {
                    "name": class_name,
                    "colorRGB": int(qupath_color_from_hex(cluster_colors_hex[color_index]))
                },
                "name": class_name,
                "clusterId": class_id,
                "exportResolutionMicrons": export_resolution_um,
                "isLocked": False,
            },
        })
    except Exception as error:
        print(f"  Warning merging class {class_id}: {error}")

raw_output_file = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_raw_macro_clusters.geojson")
file_size_mb = export_feature_collection(raw_features, raw_output_file)

# Export a single high-resolution valid-tissue polygon after excluding reviewed lint clusters.
excluded_cluster_ids = sorted(set(int(value) for value in VALID_TISSUE_EXCLUDED_CLUSTERS))
merged_valid_mask = valid_tissue_mask_2d & (~np.isin(cluster_map_2d, excluded_cluster_ids))

close_radius_px = int(np.ceil(
    MERGED_VALID_CLOSE_UM / (MICRONS_PER_PIXEL * CLUSTERING_DOWNSAMPLE)
))
if close_radius_px > 0:
    merged_valid_mask = cv2.morphologyEx(
        merged_valid_mask.astype(np.uint8), cv2.MORPH_CLOSE,
        disk(close_radius_px).astype(np.uint8)
    ) > 0
    merged_valid_mask &= valid_tissue_mask_2d

fill_hole_pixels = int(np.ceil(
    MERGED_VALID_FILL_HOLES_UM2 /
    (MICRONS_PER_PIXEL * CLUSTERING_DOWNSAMPLE) ** 2
))
if fill_hole_pixels > 0:
    merged_valid_mask = remove_small_holes(
        merged_valid_mask, area_threshold=fill_hole_pixels, connectivity=1
    )
    merged_valid_mask &= valid_tissue_mask_2d

merged_valid_shapes = [
    scaled_valid_geometry(geometry, CLUSTERING_DOWNSAMPLE)
    for geometry, value in rasterio.features.shapes(
        merged_valid_mask.astype(np.uint8), mask=merged_valid_mask,
        connectivity=8
    )
    if int(value) == 1
]
merged_valid_features = []
if merged_valid_shapes:
    merged_valid_geometry = make_valid(unary_union(merged_valid_shapes))
    if not merged_valid_geometry.is_empty:
        merged_valid_features.append({
            "type": "Feature", "id": str(uuid.uuid4()),
            "geometry": mapping(merged_valid_geometry),
            "properties": {
                "objectType": "annotation",
                "classification": {"name": "Valid_Tissue_From_Clusters", "colorRGB": -16711936},
                "name": "HighRes_Valid_Tissue_From_Clusters",
                "excludedClusterIds": excluded_cluster_ids,
                "exportResolutionMicrons": CLUSTERING_DOWNSAMPLE * MICRONS_PER_PIXEL,
                "isLocked": False,
            },
        })

merged_valid_output_file = os.path.join(
    OUTPUT_DIR, f"{SAMPLE_ID}_valid_tissue_from_clusters_highres.geojson"
)
merged_valid_size_mb = export_feature_collection(
    merged_valid_features, merged_valid_output_file
)

# Display the exact partition that is exported.
plot_map = export_label_map
cmap_custom = mcolors.ListedColormap(['#111111'] + cluster_colors_hex)
bounds = np.arange(-1.5, max(active_class_ids) + 1.5, 1)
norm_c = mcolors.BoundaryNorm(bounds, cmap_custom.N)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=120, constrained_layout=True)
axes[0].imshow(cluster_map_2d, cmap=cmap_custom, norm=norm_c, interpolation='nearest')
axes[0].set_title("Original 4x Cluster Labels", fontweight='bold')
axes[0].axis('off')
axes[1].imshow(plot_map, cmap=cmap_custom, norm=norm_c, interpolation='nearest')
axes[1].set_title(f"Exported Partition: {export_mode_label}", fontweight='bold')
axes[1].axis('off')
plt.show()

print("\n" + "=" * 72)
print(f"  HIGH-RES CLUSTER GEOJSON EXPORTED ({file_size_mb:.2f} MB)")
print(f"  File: {raw_output_file}")
print(f"  Resolution: {export_resolution_um:.3f} um/pixel; shared boundaries={GAPLESS_SHARED_BOUNDARIES}")
print(f"  HIGH-RES MERGED VALID TISSUE EXPORTED ({merged_valid_size_mb:.2f} MB)")
print(f"  File: {merged_valid_output_file}")
print(f"  Excluded clusters: {excluded_cluster_ids or 'none'}")
print("=" * 72)

---
# Phase 3 & 4: Functional Domain Annotation, Heatmap Signatures & MultiPolygon Vector Export

### Step 2.4 — Biological Domain Mapping, Signature Heatmaps & Final Vector Export

**Objective:** Map fine-grained unsupervised clusters into higher-order biological/anatomical tissue domains (e.g. Stroma, Epithelium, Tumor Core, Immune-rich microenvironments), compute merged domain signature heatmaps (Mean Normalized Intensity & Z-Score profiles), display spatial domain maps, and export final MultiPolygon GeoJSON annotations.

In [ ]:
# ============================================================================
# PHASE 3 & 4: Biological Domain Mapping, Signature Heatmaps & High-Res GeoJSON Export
# ============================================================================

cluster_to_domain = {
    0: "Epithelia",
    1: "Stroma",
    2: "Stroma",
    3: "Epithelia",
    4: "Stroma",
    5: "Stroma",
    6: "Stroma",
    7: "Stroma",
    8: "Epithelia",
    9: "Stroma",
    10: "Stroma",
    11: "Stroma",
    12: "Stroma",
    13: "Stroma",
    14: "Stroma",
    15: "Epithelia",
    16: "Epithelia",
    17: "Stroma",
    18: "Stroma",
    19: "Endothelia",
}

# --- Step 3.1: Biological Domain Pixel Mapping & Heatmap Signatures ---
domain_labels_valid = np.array([cluster_to_domain.get(c, f"Cluster_{c}") for c in labels_valid])
domain_profiles = pd.DataFrame(pixels_orig_valid, columns=keep_names).assign(domain=domain_labels_valid).groupby('domain').mean()
domain_z_profiles = pd.DataFrame(pixels_z_valid, columns=keep_names).assign(domain=domain_labels_valid).groupby('domain').mean()

print("\n" + "=" * 80)
print("  GENERATING MERGED BIOLOGICAL DOMAIN SIGNATURE HEATMAPS")
print("=" * 80)

# Heatmap 1: Merged Domain Absolute Signatures (Mean Normalized Intensity)
g_dom = sns.clustermap(
    domain_profiles,
    cmap='YlOrRd',
    figsize=(16, max(5, len(domain_profiles) * 0.5)),
    row_cluster=True if len(domain_profiles) > 1 else False,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[str(d) for d in domain_profiles.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_dom.ax_heatmap.set_xlabel('Marker', fontsize=13, fontweight='bold')
g_dom.ax_heatmap.set_ylabel('Biological Domain', fontsize=13, fontweight='bold')
g_dom.fig.suptitle('Merged Biological Domain Signatures (Mean Normalized Intensity)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

# Heatmap 2: Merged Domain Unique/Enriched Signatures (Mean Z-score)
g_dom_z = sns.clustermap(
    domain_z_profiles,
    cmap='RdBu_r',
    center=0.0,
    figsize=(16, max(5, len(domain_z_profiles) * 0.5)),
    row_cluster=True if len(domain_z_profiles) > 1 else False,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[str(d) for d in domain_z_profiles.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_dom_z.ax_heatmap.set_xlabel('Marker', fontsize=13, fontweight='bold')
g_dom_z.ax_heatmap.set_ylabel('Biological Domain', fontsize=13, fontweight='bold')
g_dom_z.fig.suptitle('Merged Biological Domain Unique/Enriched Signatures (Mean Z-score)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("\nTop-3 Dominant vs. Enriched Markers per Merged Biological Domain:")
print("=" * 80)
for dname in domain_profiles.index:
    row_int = domain_profiles.loc[dname]
    top3_int = row_int.nlargest(3)
    int_str = ", ".join([f"{m} ({v:.2f})" for m, v in top3_int.items()])

    row_z = domain_z_profiles.loc[dname]
    top3_z = row_z.nlargest(3)
    z_str = ", ".join([f"{m} (+{v:.2f})" for m, v in top3_z.items()])

    print(f"  Domain: {dname}")
    print(f"    Dominant (Intensity): {int_str}")
    print(f"    Enriched (Z-score):   {z_str}")
    print("-" * 80)

# --- Step 3.2: 2D Spatial Domain Mapping & MultiPolygon Vector Feature Construction (Exact Step 2.3 Resolution & Gapless Partition) ---
unique_active_domains = list(dict.fromkeys(cluster_to_domain.values()))

# High-contrast, distinctive palette for each domain
domain_palette = sns.color_palette("Set1", n_colors=max(3, len(unique_active_domains)))
dom_hex_map = {dname: mcolors.to_hex(domain_palette[i % len(domain_palette)]) for i, dname in enumerate(unique_active_domains)}

domain_to_id = {dname: idx for idx, dname in enumerate(unique_active_domains)}
domain_id_to_name = {idx: dname for idx, dname in enumerate(unique_active_domains)}

# Use the exact fine export_label_map from Step 2.3 if available, or cluster_map_2d
if 'export_label_map' in globals():
    base_partition = export_label_map
else:
    base_partition = cluster_map_2d

fine_domain_label_map = np.full(base_partition.shape, -1, dtype=np.int16)
for c_id, dname in cluster_to_domain.items():
    fine_domain_label_map[base_partition == c_id] = domain_to_id[dname]

# Extract fine gapless shapes at exact 1x scale (CLUSTERING_DOWNSAMPLE) matching Step 2.3
domain_shape_records = list(partition_shapes(fine_domain_label_map, domain_id_to_name))
export_scale = CLUSTERING_DOWNSAMPLE

from collections import defaultdict
polys_by_domain = defaultdict(list)
for domain_id, domain_name, geometry in domain_shape_records:
    try:
        polygon = scaled_valid_geometry(geometry, export_scale)
        if not polygon.is_empty:
            polys_by_domain[domain_name].append(polygon)
    except Exception:
        continue

domain_features = []
for dname in unique_active_domains:
    polys = polys_by_domain.get(dname, [])
    if not polys: continue
    try:
        merged_geom = make_valid(unary_union(polys))
        if merged_geom.is_empty: continue

        # Keep shared gapless boundaries (disable independent polish_boundary in gapless mode)
        if not GAPLESS_SHARED_BOUNDARIES:
            merged_geom = polish_boundary(merged_geom, BOUNDARY_SIMPLIFY_UM, BOUNDARY_SMOOTH_UM, MICRONS_PER_PIXEL, BOUNDARY_SMOOTH_QUAD_SEGS)
        if merged_geom.is_empty: continue

        hex_c = dom_hex_map[dname]
        domain_features.append({
            "type": "Feature", "id": str(uuid.uuid4()),
            "geometry": mapping(merged_geom),
            "properties": {
                "objectType": "annotation",
                "classification": {"name": dname, "colorRGB": int(qupath_color_from_hex(hex_c))},
                "name": dname,
                "isLocked": False,
            },
        })
    except Exception as e:
        print(f"  Warning merging domain '{dname}': {e}")

domain_output_file = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_macro_domains.geojson")
dom_size_mb = export_feature_collection(domain_features, domain_output_file)

# --- Step 3.3: Visualizing Merged Biological Domain Map & GeoJSON Features ---
fig, ax = plt.subplots(figsize=(12, 10), dpi=120)
cmap_colors = [dom_hex_map[dname] for dname in unique_active_domains]
if len(cmap_colors) > 0:
    cmap_dom = mcolors.ListedColormap(cmap_colors)
    bounds = np.arange(-0.5, len(unique_active_domains) + 0.5, 1)
    norm_dom = mcolors.BoundaryNorm(bounds, cmap_dom.N)

    masked_grid = np.ma.masked_equal(fine_domain_label_map, -1)
    im = ax.imshow(masked_grid, cmap=cmap_dom, norm=norm_dom, interpolation='nearest')

    patches = [mpatches.Patch(color=dom_hex_map[dname], label=dname) for dname in unique_active_domains]
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', title="Biological Domains", frameon=True)

ax.set_title(f"Merged Biological Domains & Vector Export (Fine 1x Resolution) — {SAMPLE_ID}", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("\n" + "=" * 65)
print(f"  COMPLETE PIPELINE — HIGH-RES GAPLESS DOMAIN GEOJSON EXPORT ({dom_size_mb:.2f} MB)")
print("=" * 65)
print(f"  ✓ Final Macro Domains GeoJSON : {domain_output_file}")
print(f"  Resolution: {export_scale * MICRONS_PER_PIXEL:.3f} um/pixel; shared boundaries=True")
print("=" * 65)


---
# Phase 5: Independent GeoJSON & ROI Image Analysis (QuPath Integration)

### Step 3.1 — Standalone Heatmap & Spatial Annotation Inspector

**Objective:** Independently load any GeoJSON annotation file (e.g. modified or exported from QuPath) and corresponding OME-TIFF ROI image. Automatically extracts biomarker features, computes domain-level intensity and Z-score heatmaps, and renders the spatial domain map with annotations and legends.

*Note: This cell is fully self-contained and can be executed independently of the rest of the notebook. Simply modify `INPUT_GEOJSON_PATH` and `INPUT_TIFF_PATH` to inspect any dataset.*

In [ ]:
# ============================================================================
# PHASE 5: STANDALONE GEOJSON & ROI IMAGE HEATMAP & SPATIAL INSPECTOR
# ============================================================================
# This cell can be executed independently of the rest of the notebook.
# Modify INPUT_GEOJSON_PATH and INPUT_TIFF_PATH below to analyze any image
# and corresponding GeoJSON file (e.g., edited in QuPath).
# ============================================================================

import os
import sys
import json
import uuid
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import tifffile
import cv2

from scipy.ndimage import gaussian_filter
from skimage.filters import threshold_otsu

from shapely.geometry import Polygon, MultiPolygon, shape, mapping
from shapely.ops import unary_union
from shapely.validation import make_valid

import rasterio.features

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

# ----- 1. Configuration & Input File Paths (Standalone Execution) -----
# Directly set your sample ID, image directory, and output directory here for independent execution.
SAMPLE_ID  = "HCC_SARC_1.0_CODEX_multiround_merged"
ROI_DIR    = "/project/SCCC/Aguilera_lab/shared/SPACEMAP_v15/project/HCC_joshtheriot/outputs/registered"
OUTPUT_DIR = "/project/SCCC/Aguilera_lab/shared/SPACEMAP_v15/project/HCC_joshtheriot/outputs/registered/tissue_classification/HCC_SARC_1"

# Specify the target GeoJSON and OME-TIFF image paths directly:
# (You can edit INPUT_GEOJSON_PATH to point to any GeoJSON file, e.g., hand_annotated.geojson)
INPUT_GEOJSON_PATH = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_hand_annotated.geojson")
INPUT_TIFF_PATH    = os.path.join(ROI_DIR, f"{SAMPLE_ID}.ome.tiff")

# Dynamic fallback: if hand_annotated.geojson doesn't exist, check macro_domains.geojson or any GeoJSON in OUTPUT_DIR
if not os.path.exists(INPUT_GEOJSON_PATH):
    alt_domains_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_macro_domains.geojson")
    if os.path.exists(alt_domains_path):
        INPUT_GEOJSON_PATH = alt_domains_path
    elif os.path.exists(OUTPUT_DIR):
        possible_geojsons = [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.endswith(".geojson")]
        if possible_geojsons:
            INPUT_GEOJSON_PATH = possible_geojsons[0]

# Dynamic fallback: if default TIFF path doesn't exist, search ROI_DIR for matching .ome.tiff
if not os.path.exists(INPUT_TIFF_PATH) and os.path.exists(ROI_DIR):
    possible_tiffs = [os.path.join(ROI_DIR, f) for f in os.listdir(ROI_DIR) if f.endswith(".tiff") or f.endswith(".tif")]
    if possible_tiffs:
        INPUT_TIFF_PATH = possible_tiffs[0]

# Spatial downsampling & signal processing settings
ANALYSIS_DOWNSAMPLE  = 4      # 4x spatial downsampling for fast feature processing
ARCSINH_COFACTOR     = 5.0    # Cofactor for arcsinh signal normalization
RANDOM_SEED          = 42

EXCLUDE_CHANNELS_LIST = ["HE_Red", "HE_Green", "HE_Blue"]

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
})
sns.set_style("whitegrid")

# Helper function to generate distinctive deterministic colors using random seed 42 (ignoring GeoJSON colors)
def generate_domain_hex_color(name, seed=RANDOM_SEED):
    if name in ["Low_Signal_Tissue", "Background", "Non-Tissue"]:
        return "#888888"
    hash_bytes = hashlib.md5(f"{seed}_{name}".encode('utf-8')).digest()
    hash_int = int.from_bytes(hash_bytes[:4], 'big')
    hue = (hash_int % 360) / 360.0
    sat = 0.75 + 0.25 * ((hash_int >> 8) % 100) / 100.0
    val = 0.85 + 0.15 * ((hash_int >> 16) % 100) / 100.0
    rgb = mcolors.hsv_to_rgb((hue, sat, val))
    return mcolors.to_hex(rgb)

print("=" * 80)
print("  STANDALONE GEOJSON & ROI IMAGE ANALYSIS")
print("=" * 80)
print(f"  GeoJSON File : {INPUT_GEOJSON_PATH}")
print(f"  TIFF Image   : {INPUT_TIFF_PATH}")

assert os.path.exists(INPUT_GEOJSON_PATH), f"GeoJSON file not found: {INPUT_GEOJSON_PATH}"
assert os.path.exists(INPUT_TIFF_PATH), f"TIFF file not found: {INPUT_TIFF_PATH}"

# ----- 2. Load OME-TIFF Multichannel Image Stack -----
with tifffile.TiffFile(INPUT_TIFF_PATH) as tif:
    ome_xml = tif.ome_metadata
    root = ET.fromstring(ome_xml)
    ns = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
    all_channel_names = [
        ch.get('Name', f'Channel_{i}')
        for i, ch in enumerate(root.findall('.//ome:Channel', ns))
    ]
    n_channels = len(all_channel_names)
    n_series = len(tif.series)
    n_levels_per_channel = n_series // n_channels if n_channels > 0 else 1
    is_pyramidal = n_levels_per_channel > 1

    # Filter biomolecular channels
    keep_indices = []
    keep_names = []
    for idx, cname in enumerate(all_channel_names):
        if not any(ex.lower() in cname.lower() for ex in EXCLUDE_CHANNELS_LIST):
            keep_indices.append(idx)
            keep_names.append(cname)

    print(f"  ✓ Image channels detected: {n_channels} total ({len(keep_names)} biomarker channels selected)")

    # Function to read channel at downsample level
    def read_channel_ds(tiff_obj, ch_idx, factor):
        if is_pyramidal and factor > 1:
            lvl = min(int(np.log2(factor)), n_levels_per_channel - 1)
            s_idx = ch_idx * n_levels_per_channel + lvl
            return tiff_obj.series[s_idx].asarray().astype(np.float32)
        else:
            return tiff_obj.pages[ch_idx].asarray()[::factor, ::factor].astype(np.float32)

    img_channels = []
    for ch_i in keep_indices:
        img_channels.append(read_channel_ds(tif, ch_i, ANALYSIS_DOWNSAMPLE))

img_stack = np.stack(img_channels, axis=0)  # Shape: (C, H_down, W_down)
C, H_down, W_down = img_stack.shape
print(f"  ✓ Downsampled image feature stack loaded: {H_down}×{W_down} px ({C} channels)")

# ----- 3. Arcsinh Normalization & Z-Score Feature Standardization -----
print("  Applying arcsinh normalization & Z-score scaling...")
pixels_orig = img_stack.reshape(C, -1).T  # Shape: (H*W, C)
pixels_arcsinh = np.arcsinh(pixels_orig / ARCSINH_COFACTOR)

# Perform Z-score standardization per marker channel
mean_vals = np.mean(pixels_arcsinh, axis=0)
std_vals = np.std(pixels_arcsinh, axis=0)
std_vals[std_vals == 0] = 1.0
pixels_z = (pixels_arcsinh - mean_vals) / std_vals

# ----- 4. Load & Parse GeoJSON Features & Domain Geometries -----
with open(INPUT_GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

features = geojson_data.get("features", [])
print(f"  ✓ GeoJSON loaded: {len(features)} feature annotations found.")

# Group polygons by domain class name
domain_geoms = {}
domain_color_map = {}

for feat in features:
    props = feat.get("properties", {})
    geom_dict = feat.get("geometry", {})
    if not geom_dict: continue

    # Extract classification name
    classification = props.get("classification", {})
    if isinstance(classification, dict):
        dname = classification.get("name", props.get("name", "Unclassified"))
        color_val = classification.get("colorRGB", None)
    else:
        dname = str(classification) if classification else props.get("name", "Unclassified")
        color_val = None

    # Generate distinctive color using RANDOM_SEED = 42 (ignoring assigned GeoJSON colors)
    if dname not in domain_color_map:
        domain_color_map[dname] = generate_domain_hex_color(dname, seed=RANDOM_SEED)

    # Convert geometry & scale to downsampled grid coordinates
    geom_shape = shape(geom_dict)
    if geom_shape.is_empty: continue

    # Scale geometry down to match ANALYSIS_DOWNSAMPLE grid
    scale_factor = 1.0 / ANALYSIS_DOWNSAMPLE
    from shapely.affinity import scale
    scaled_geom = scale(geom_shape, xfact=scale_factor, yfact=scale_factor, origin=(0, 0))

    if dname not in domain_geoms:
        domain_geoms[dname] = []
    domain_geoms[dname].append(scaled_geom)

unique_domains = list(domain_geoms.keys())
print(f"  ✓ Identified {len(unique_domains)} unique biological domains: {unique_domains}")

# ----- 5. Rasterize GeoJSON Domain Polygons onto Image Grid -----
domain_mask = np.full((H_down, W_down), -1, dtype=int)

for dom_idx, dname in enumerate(unique_domains):
    geoms_list = domain_geoms[dname]
    if not geoms_list: continue
    shapes_tuples = [(g, dom_idx) for g in geoms_list if not g.is_empty]
    if shapes_tuples:
        raster = rasterio.features.rasterize(
            shapes_tuples,
            out_shape=(H_down, W_down),
            fill=-1,
            dtype=np.int32
        )
        domain_mask[raster == dom_idx] = dom_idx

flat_mask = domain_mask.ravel()
valid_domain_pixels = flat_mask >= 0
print(f"  ✓ GeoJSON domain rasterization complete. {np.sum(valid_domain_pixels):,} pixels ({100*np.sum(valid_domain_pixels)/len(flat_mask):.1f}% coverage)")

# ----- 6. Compute Domain Signatures & Plot Heatmaps -----
domain_labels = [unique_domains[idx] for idx in flat_mask[valid_domain_pixels]]
pixels_orig_sub = pixels_arcsinh[valid_domain_pixels]
pixels_z_sub = pixels_z[valid_domain_pixels]

domain_profiles = pd.DataFrame(pixels_orig_sub, columns=keep_names).assign(domain=domain_labels).groupby('domain').mean()
domain_z_profiles = pd.DataFrame(pixels_z_sub, columns=keep_names).assign(domain=domain_labels).groupby('domain').mean()

print("\n" + "=" * 80)
print("  HEATMAP 1: ABSOLUTE BIOMARKER SIGNATURES (Mean Normalized Intensity)")
print("=" * 80)

g_dom = sns.clustermap(
    domain_profiles,
    cmap='YlOrRd',
    figsize=(16, max(5, len(domain_profiles) * 0.5)),
    row_cluster=True if len(domain_profiles) > 1 else False,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[str(d) for d in domain_profiles.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_dom.ax_heatmap.set_xlabel('Biomarker', fontsize=13, fontweight='bold')
g_dom.ax_heatmap.set_ylabel('GeoJSON Domain / Annotation', fontsize=13, fontweight='bold')
g_dom.fig.suptitle('Standalone GeoJSON Domain Signatures (Mean Normalized Intensity)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("\n" + "=" * 80)
print("  HEATMAP 2: UNIQUE / ENRICHED SIGNATURES (Mean Z-score)")
print("=" * 80)

g_dom_z = sns.clustermap(
    domain_z_profiles,
    cmap='RdBu_r',
    center=0.0,
    figsize=(16, max(5, len(domain_z_profiles) * 0.5)),
    row_cluster=True if len(domain_z_profiles) > 1 else False,
    col_cluster=True,
    linewidths=0.5,
    linecolor='white',
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(0.02, 0.80, 0.03, 0.15),
    yticklabels=[str(d) for d in domain_z_profiles.index],
    xticklabels=True,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7.5},
)
g_dom_z.ax_heatmap.set_xlabel('Biomarker', fontsize=13, fontweight='bold')
g_dom_z.ax_heatmap.set_ylabel('GeoJSON Domain / Annotation', fontsize=13, fontweight='bold')
g_dom_z.fig.suptitle('Standalone GeoJSON Domain Enriched Signatures (Mean Z-score)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("\nTop-3 Dominant vs. Enriched Markers per GeoJSON Domain:")
print("=" * 80)
for dname in domain_profiles.index:
    row_int = domain_profiles.loc[dname]
    top3_int = row_int.nlargest(3)
    int_str = ", ".join([f"{m} ({v:.2f})" for m, v in top3_int.items()])

    row_z = domain_z_profiles.loc[dname]
    top3_z = row_z.nlargest(3)
    z_str = ", ".join([f"{m} (+{v:.2f})" for m, v in top3_z.items()])

    print(f"  GeoJSON Domain: {dname}")
    print(f"    Dominant (Intensity): {int_str}")
    print(f"    Enriched (Z-score):   {z_str}")
    print("-" * 80)

# ----- 7. Display Spatial GeoJSON Overlay Map with Colors & Annotations -----
print("\n  Rendering Spatial GeoJSON Domain Overlay Map...")

cmap_colors = [domain_color_map[dname] for dname in unique_domains]

fig, ax = plt.subplots(figsize=(12, 10))
if len(cmap_colors) > 0:
    cmap_dom = mcolors.ListedColormap(cmap_colors)
    bounds = np.arange(-0.5, len(unique_domains) + 0.5, 1)
    norm_dom = mcolors.BoundaryNorm(bounds, cmap_dom.N)

    masked_mask = np.ma.masked_equal(domain_mask, -1)
    im = ax.imshow(masked_mask, cmap=cmap_dom, norm=norm_dom, interpolation='nearest')

    patches = [mpatches.Patch(color=domain_color_map[dname], label=dname) for dname in unique_domains]
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', title="GeoJSON Annotations", frameon=True)

ax.set_title(f"Standalone GeoJSON Spatial Domain Map — {os.path.basename(INPUT_GEOJSON_PATH)}", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("  ✓ STANDALONE GEOJSON & ROI IMAGE ANALYSIS COMPLETE")
print("=" * 80)